# Réplica de Aradillas (2018) — ENIGH 2022
## Poder de mercado y bienestar social en hogares mexicanos

**Referencia:** Aradillas López, A. (2018). *Estudio sobre el impacto que tiene el poder de
mercado en el bienestar de los hogares mexicanos*. COFECE, México.

**Modelo:** Sistema EASI (*Exact Affine Stone Index*) de Lewbel & Pendakur (2009).

---

### Estructura del notebook

| Sección | Contenido | Cuadros del paper |
|---------|-----------|-------------------|
| 1 | Construcción de índices de precios por ciudad (46 ciudades, INPC/INPP) | — |
| 2 | Carga y filtrado de microdatos ENIGH 2014 | Cuadro 2 |
| 3 | Gastos por categoría, índices Divisia, variables Z | — |
| 4 | Sistema aproximado de demanda (OLS iterado, 16 pasos) | — |
| 5 | Matrices de parámetros, residuos ε_h, utilidad indirecta exacta | — |
| 6 | Demandas Marshallianas y elasticidades por ciudad y región | Cuadros 4, 5 |
| 7 | Markups por categoría y ciudad (modelo NEIO) | Cuadros 8, 9 |
| 8 | Variación equivalente, pérdida de bienestar por decil y Gini | Cuadro 10 |

---

### Decisiones metodológicas clave (diferencias con código Gauss original)

1. **Filtro de tenencia de vivienda:** El programa Gauss 2014 usa códigos 3 y 4
   (vivienda propia pagándose + totalmente pagada). El programa de 2006 usaba 4 y 5.
   Esta diferencia explica ~1,757 hogares de diferencia en la muestra pre-trim.

2. **Trim iterativo:** 1% en cada cola por iteración (16 iteraciones). La muestra
   pasa de 12,372 a 8,940 hogares. El paper reporta 15,586 hogares (muestra pre-trim
   con filtros menos restrictivos no completamente replicados).

3. **Utilidad indirecta exacta:** El Gauss usa `optmum()` (Newton-Raphson interno).
   Replicamos con Newton-Raphson con damping (paso máximo = 2.0) + fallback a
   `minimize_scalar` bounded. Converge en ~66% de hogares via Newton; el resto via fallback.

4. **Epsilon (residuos):** Se extrae directamente de la última iteración OLS
   (con Y ajustado por simetría), igual que en Gauss. NO se recomputa externamente.

5. **Demandas agregadas:** Ponderadas por factor de expansión π_h (col 7 del
   concentrado ENIGH), replicando la ecuación del paper: Q^M = Σ q_h · π_h.

6. **Precios para markups:** Construidos desde P_46[producto] (en pesos MXN,
   deflactados desde junio 2011) con shares de subproductos del gasto observado.
   NO desde exp(precios_matrix_ln) que es un índice normalizado, no pesos.

7. **Brecha de muestra:** 8,940 (réplica) vs 15,586 (paper). Causa: filtros de
   muestra ligeramente distintos + trim acumulado. Consecuencia: elasticidades
   comprimidas hacia 1.0 (MAE=0.207 vs Cuadro 4). Cuadro 5 (regiones): réplica
   exacta (8/8 dentro de ±0.15). Cuadro 10 (bienestar): patrón cualitativo correcto.

---

### Resultados comparativos

| Resultado | Réplica | Paper | Diferencia |
|-----------|---------|-------|------------|
| Cuadro 5: elasticidades regionales | 8/8 ✓ | — | < ±0.15 en todas |
| VE/ingreso media nacional | 14.3% | 15.7% | -9% |
| Regresividad decil I / decil X | 5.9x | 4.42x | +33% |
| Gini reducción sin poder mercado | 5.6% | 7.3% | -23% |
| β_η Pan | 1.020 | 1.477 | -31% |
| β_η Autobús foráneo | 0.084 | 0.081 | +4% |


## 0. Instalación de dependencias y carga de archivos

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Instalar scipy para distancias y optimización
# !pip install -q scipy numpy pandas

In [3]:
import numpy as np
import pandas as pd
import os
import math
from scipy.optimize import minimize_scalar
import warnings
import unicodedata
import re
import glob
import sys
import json
warnings.filterwarnings('ignore')

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


In [4]:
# ============================================================
# INSTRUCCIONES:
# Sube todos los archivos .asc a Google Colab usando el panel
# de archivos (ícono de carpeta a la izquierda) o ejecuta:
#   from google.colab import files
#   files.upload()
# y sube los archivos .asc uno por uno.
#
# Alternativamente, si los tienes en Google Drive:
#   from google.colab import drive
#   drive.mount('/content/drive')
# y ajusta DATA_DIR abajo.
# ============================================================

DATA_DIR = 'Replica_COFECE/Data_2022/'   # Ajusta si usas Drive, p.ej. '/content/drive/MyDrive/aradillas/'

print(f'Directorio de datos: {DATA_DIR}')

Directorio de datos: Replica_COFECE/Data_2022/


In [5]:
sys.path.append("./Replica_COFECE/Codigo")  


## 1. Carga y construcción de precios locales

El modelo usa precios de referencia de junio 2011 (46 ciudades) deflactados
al período de levantamiento del ENIGH 2014 (agosto–noviembre 2014)
usando índices INPC por subgénero y ciudad.

Para cada ciudad $i$ y producto $j$:
$$P_{ij,2014} = P_{ij,\text{jun2011}} \times \text{mediana}\left(\frac{\text{INPC}_{ij,t}}{\text{INPC}_{ij,\text{jun2011}}}\right), \quad t \in [\text{ago2014, nov2014}]$$

### Sección 1 — Índices de precios por ciudad

**Fuentes de datos:**
- `data_ciudades/inpc_{ciudad}.csv` (): Series mensuales INPC del INEGI para 55 ciudades y 53 subgéneros de precios. Periodo cubierto: 2018- 2026. https://www.inegi.org.mx/programas/inpc/2018a/#tabulados
- `inpc_46_ciudades.asc` (4,968 × 66): Series mensuales INPC del INEGI para 46 ciudades
  y 61 subgéneros de precios. Período cubierto: 2002–2021. https://www.inegi.org.mx/app/preciospromedio/
- `inpp_construccion_46_ciudades.asc` (4,968 × 6): Índice de precios al productor para
  materiales de construcción. Se usa como proxy de precio para la categoría 12 (materiales). -> en `viviendas.csv` estan los materiales de construccion
- `precios_promedio_46_ciudades_junio_2011.asc` (46 × 70): Precios promedio observados en
  junio 2011 para 63 productos específicos en las 46 ciudades. Fuente: INEGI. Estos son los
  precios de referencia base.

**Procedimiento de deflactación:**
Para cada ciudad *i* y producto *j*:
$$P_{ij,2014} = P_{ij,\text{jun2011}} \times \text{mediana}\left(\frac{\text{INPC}_{ij,t}}{\text{INPC}_{ij,\text{jun2011}}}\right), \quad t \in [\text{ago-2014, nov-2014}]$$

La mediana sobre agosto–noviembre 2014 corresponde al período de levantamiento de la ENIGH 2014.
El uso de la mediana (vs la media) es más robusto a choques de precios puntuales.

**Resultado:** Matriz `P_46[producto]` con shape (46,) para cada producto — precios en pesos
MXN a precios de agosto–noviembre 2014, para las 46 ciudades del sistema INPC.

**Ciudades:** Las 46 ciudades del sistema INPC del INEGI (ver Cuadro 3 del paper).
Los mercados geográficos se agrupan en 8 regiones para el análisis regional.


La series del Inegi vienen con columnas que no nos interesan para este estudio. Estas son las columnas que nos interesan, ya que nos quedamos con las mismas que las que se definieron en el estudio original

In [6]:
from inpc_lista_productos import target_columns
print(target_columns[0:5])  # Muestra las primeras 5 columnas objetivo

['Índice Nacional de Precios al Consumidor. Base segunda quincena Julio 2018. Actualización de Canasta y Ponderadores 2024 (mensual), Por ciudad, 7. Area Metropolitana de la Cd. de México, Índice de precios al consumidor, por objeto del gasto, Índice general, 1. Alimentos, bebidas y tabaco, 1.1. Alimentos, 1.1.1. Pan, tortillas y cereales, 01 Tortillas y derivados del maíz, 014 Tortilla de maíz', 'Índice Nacional de Precios al Consumidor. Base segunda quincena Julio 2018. Actualización de Canasta y Ponderadores 2024 (mensual), Por ciudad, 7. Area Metropolitana de la Cd. de México, Índice de precios al consumidor, por objeto del gasto, Índice general, 1. Alimentos, bebidas y tabaco, 1.1. Alimentos, 1.1.1. Pan, tortillas y cereales, 02 Pan, 008 Pan blanco', 'Índice Nacional de Precios al Consumidor. Base segunda quincena Julio 2018. Actualización de Canasta y Ponderadores 2024 (mensual), Por ciudad, 7. Area Metropolitana de la Cd. de México, Índice de precios al consumidor, por objeto de

In [7]:
# Extraemos SOLO la parte final (ej: "014 Tortilla de maíz")
# Esto limpia nuestra lista de búsqueda de toda la información previa, y nos evita tener que cambiar de filtro por ciudad
productos_objetivo = [col.split(',')[-1].strip() for col in target_columns]
print(productos_objetivo)

['014 Tortilla de maíz', '008 Pan blanco', '010 Pan dulce', '022 Pollo', '018 Carne de res', '025 Vísceras de res', '020 Chorizo', '021 Jamón', '023 Salchichas', '024 Tocino', '034 Leche pasteurizada y fresca', '032 Leche en polvo', '033 Leche evaporada y condensada', '030 Crema y otros productos a base de leche', '036 Queso amarillo', '037 Queso fresco', '038 Queso manchego y Chihuahua', '039 Queso Oaxaca y asadero', '044 Mantequilla', '031 Huevo', '045 Aguacate', '047 Guayaba', '048 Limón', '049 Manzana', '050 Melón', '051 Naranja', '052 Papaya', '054 Piña', '055 Plátanos', '056 Sandía', '057 Uva', '060 Calabacita', '061 Cebolla', '062 Chayote', '063 Chile poblano', '065 Chile serrano', '067 Ejotes', '070 Jitomate', '071 Lechuga y col', '072 Nopales', '073 Papa y otros tubérculos', '075 Pepino', '076 Tomate verde', '078 Zanahoria', '068 Frijol', '095 Jugos o néctares envasados', '099 Agua embotellada', '100 Refrescos envasados', '184 Analgésicos', '185 Antibióticos', '186 Antigripale

In [8]:
def cargar_csv_filtro(path_file: str) -> tuple:
    """
    Detecta metadatos, filtra columnas usando solo el nombre del producto 
    y renombra las columnas finales.
    """
    nom_df = os.path.splitext(os.path.basename(path_file))[0]
    
    # 1. Detección robusta de la línea de encabezado
    with open(path_file, encoding='latin-1') as f:
        raw_lines = f.readlines()
    
    header_idx = max(range(len(raw_lines)), key=lambda i: raw_lines[i].count(','))
    
    # 2. Filtro de las columnas que nos interesan
    # Verificamos si la columna limpia (strip) termina con alguno de nuestros productos_objetivo
    filtro_columnas = lambda col_name: any(col_name.strip().endswith(prod) for prod in productos_objetivo)
    
    # 3. Lectura optimizada
    df = pd.read_csv(
        path_file, 
        encoding='latin-1', 
        skiprows=header_idx, 
        usecols=filtro_columnas,
        engine='python'
    )
    
    # 4. Renombrado de columnas
    df = df.rename(columns=lambda x: x.split(',')[-1].strip())
    
    # 5. Drop de las dos primeras líneas, que no forman parte de los índices
    df = df.iloc[2:].reset_index(drop=True)
    
    return nom_df, df

In [9]:
# ==========================================
# EJECUCIÓN EN BUCLE PARA TODAS LAS CIUDADES
# ==========================================
# 1. Obtener la lista de todos los archivos CSV en la carpeta
# (Ajusta la extensión '.CSV' o '.csv' según dicten tus archivos reales)
ciudades = glob.glob(DATA_DIR + 'data_ciudades/*.CSV')

# 2. Diccionario centralizado para guardar los DataFrames 
ciudades_inpc = {}

print(f"Se encontraron {len(ciudades)} archivos de ciudades para procesar.\n")

for file_path in ciudades:
    try:
        # Ejecución de tu función optimizada
        nombre_ciudad, df_ciudad = cargar_csv_filtro(file_path)
        
        # Guardar en el diccionario usando el nombre de la ciudad como clave
        ciudades_inpc[nombre_ciudad] = df_ciudad
        
        
        #print(f"✓ {nombre_ciudad}: Cargado con éxito ({len(df_ciudad.columns)} columnas, {len(df_ciudad)} filas).")
        
    except Exception as e:
        print(f"✗ Error al procesar el archivo {os.path.basename(file_path)}: {e}")

print("\n¡Procesamiento completo!")

Se encontraron 55 archivos de ciudades para procesar.


¡Procesamiento completo!


In [10]:
print(ciudades_inpc['inpc_acapulco'].columns)

Index(['014 Tortilla de maíz', '008 Pan blanco', '010 Pan dulce', '022 Pollo',
       '018 Carne de res', '025 Vísceras de res', '020 Chorizo', '021 Jamón',
       '023 Salchichas', '024 Tocino', '034 Leche pasteurizada y fresca',
       '032 Leche en polvo', '033 Leche evaporada y condensada',
       '030 Crema y otros productos a base de leche', '036 Queso amarillo',
       '037 Queso fresco', '038 Queso manchego y Chihuahua',
       '039 Queso Oaxaca y asadero', '044 Mantequilla', '031 Huevo',
       '045 Aguacate', '047 Guayaba', '048 Limón', '049 Manzana', '050 Melón',
       '051 Naranja', '052 Papaya', '054 Piña', '055 Plátanos', '056 Sandía',
       '057 Uva', '060 Calabacita', '061 Cebolla', '062 Chayote',
       '063 Chile poblano', '065 Chile serrano', '067 Ejotes', '070 Jitomate',
       '071 Lechuga y col', '072 Nopales', '073 Papa y otros tubérculos',
       '075 Pepino', '076 Tomate verde', '078 Zanahoria', '068 Frijol',
       '095 Jugos o néctares envasados', '099 Agua

In [11]:
archivos_inp_pp = sorted(glob.glob(DATA_DIR + 'data_inp_pp/*.CSV'))

primer_archivo = archivos_inp_pp[0]

# --- PASO 1: Leer el primer archivo para establecer la estructura base ---
with open(primer_archivo, encoding='latin-1') as f:
    lines_1 = f.readlines()

header_idx_1 = max(range(len(lines_1)), key=lambda i: lines_1[i].count(','))

df1 = pd.read_csv(
    primer_archivo, 
    encoding='latin-1', 
    skiprows=header_idx_1, 
    engine='python'
)
df1.columns = df1.columns.str.strip().str.lower()

# Esta lista será nuestro molde obligatorio para todos los demás archivos
columnas_reales = df1.columns.tolist()

# Inicializamos la lista de dataframes con el primero ya limpio
dataframes = [df1]


# --- PASO 2: Procesar dinámicamente los archivos restantes (Índices 1 y 2) ---
for archivo in archivos_inp_pp[1:]:
    with open(archivo, encoding='latin-1') as f:
        lines_temp = f.readlines()
        
    if not lines_temp:
        print(f"Advertencia: El archivo {archivo} está vacío.")
        continue
        
    header_idx_temp = max(range(len(lines_temp)), key=lambda i: lines_temp[i].count(','))
    
    # Intento de lectura estándar
    df_temp = pd.read_csv(
        archivo, 
        encoding='latin-1', 
        skiprows=header_idx_temp, 
        engine='python'
    )
    df_temp.columns = df_temp.columns.str.strip().str.lower()
    
    # Si las columnas no coinciden con nuestro molde, forzamos la estructura
    if set(df_temp.columns) != set(columnas_reales):
        df_temp = pd.read_csv(
            archivo, 
            encoding='latin-1', 
            skiprows=header_idx_temp, 
            header=0,               # Ignora el encabezado erróneo del archivo
            names=columnas_reales,  # Fuerza a usar las columnas del archivo 1
            engine='python'
        )
    
    # Agregamos el dataframe verificado a la lista
    dataframes.append(df_temp)


# --- PASO 3: Concatenar todos con total seguridad ---
df_precios_promedios = pd.concat(dataframes, ignore_index=True)

print(f"Proceso completado. Registros totales: {len(df_precios_promedios)}")

Proceso completado. Registros totales: 19861


In [12]:
print(df_precios_promedios)

       2018  07  17/08/2018 12:00:00 a. m.  43    campeche, camp.  \
0      2018   7  17/08/2018 12:00:00 a. m.  43    Campeche, Camp.   
1      2018   7  17/08/2018 12:00:00 a. m.  43    Campeche, Camp.   
2      2018   7  17/08/2018 12:00:00 a. m.  43    Campeche, Camp.   
3      2018   7  17/08/2018 12:00:00 a. m.  43    Campeche, Camp.   
4      2018   7  17/08/2018 12:00:00 a. m.  43    Campeche, Camp.   
...     ...  ..                        ...  ..                ...   
19856  2018   7  17/08/2018 12:00:00 a. m.   4  Guadalajara, Jal.   
19857  2018   7  17/08/2018 12:00:00 a. m.   4  Guadalajara, Jal.   
19858  2018   7  17/08/2018 12:00:00 a. m.   4  Guadalajara, Jal.   
19859  2018   7  17/08/2018 12:00:00 a. m.   4  Guadalajara, Jal.   
19860  2018   7  17/08/2018 12:00:00 a. m.   4  Guadalajara, Jal.   

        1. alimentos, bebidas y tabaco            1.1. alimentos  \
0       1. Alimentos, bebidas y tabaco            1.1. Alimentos   
1       1. Alimentos, bebidas y tab

In [13]:
# Renombrar las columnas para mejorar lisibilidad
nuevos_nombres = [
    'Year',       # Columna 1 (antigua '# 2018')
    'Month',        # Columna 2 (antigua '# 07')
    'Date',      # Columna 3 (antigua '17/08/2018...')
    'Id_city',  # Columna 4 (antigua '# 43')
    'City_name',  # Columna 5 (antigua 'campeche, can')
    'Category',  # Columna 6 (antigua '1. alimentos, bebidas y tabaco')
    'Sub_category',  # Columna 7 (antigua ' 1.1. Alimentos')
    'Group',  # Columna 8 (antigua ' 1.1.4. Leche, derivados de leche y huevo')
    'Sub_group',  # Columna 9 (antigua ' 12 leche procesada')
    'Id_class',  # Columna 10 (antigua '032')
    'Class',  # Columna 11 (antigua 'leche evaporada, condensada y maternizada')
    'Id_product',  # Columna 12 (antigua '006')
    'Product_name',  # Columna 13 (antigua 'nestle, maternizada, nan, optipro, et 2, lata de 1200 gr')
    'Price',  # Columna 14 (antigua '249.17')
    'Quantity',  # Columna 15 (antigua '1')
    'Unit',  # Columna 16 (antigua 'kg')
    'No_data',  # Columna 17 (antigua 'unnamed: 16')
]

# Attribución de los nuevos nombres
df_precios_promedios.columns = nuevos_nombres
df_precios_promedios = df_precios_promedios.drop(columns=['No_data'])  # Eliminamos la columna innecesaria

In [14]:
#Construcción

file_path = DATA_DIR + 'inpp_construccion.csv'

# 1. Lectura
with open(file_path, encoding='latin-1') as f:
    raw_lines = f.readlines()

header_idx = max(range(len(raw_lines)), key=lambda i: raw_lines[i].count(','))

df_construccion = pd.read_csv(
    file_path, 
    encoding='latin-1', 
    skiprows=header_idx, 
    engine='python'
)

# 2. Función precisa de nombres
def extraer_ciudad_y_concepto(nombre_columna):
    col_str = str(nombre_columna).strip()
    if "(antes INCEVIS)," in col_str:
        subseccion = col_str.split("(antes INCEVIS),")[-1]
        ubicacion = subseccion.split(",")[0].strip().lower()
    elif "Área Metropolitana de la Cd. de México" in col_str:
        ubicacion = "área metropolitana de la cd. de méxico"
    elif "Nacional" in col_str:
        ubicacion = "nacional"
    else:
        ubicacion = col_str.split(",")[0].strip().lower()

    col_lower = col_str.lower()
    if "alquiler de maquinaria" in col_lower:
        concepto = "alquiler_maquinaria"
    elif "mano de obra" in col_lower:
        concepto = "mano_obra"
    elif "materiales de construcción" in col_lower:
        concepto = "materiales"
    else:
        concepto = "general"
        
    if ubicacion in ["título", "titulo", "concepto", "fecha"]:
        return "fecha_o_titulo"
        
    return f"{ubicacion}_{concepto}"

# 3. Asignar nombres a las columnas
df_construccion.columns = [extraer_ciudad_y_concepto(c) for c in df_construccion.columns]

# --- AQUÍ ESTÁ EL CAMBIO DE LIMPIEZA SEGURO ---
# Si la primera fila contiene códigos como '166002.0', la eliminamos usando su posición
if '16600' in str(df_construccion.iloc[0, 1]):
    df_construccion_test = df_construccion.iloc[1:].reset_index(drop=True)

# 4. Filtrar SOLO las columnas de materiales
columnas_materiales = [
    col for col in df_construccion.columns 
    if col == 'fecha_o_titulo' or col.endswith('_materiales')
]

df_materiales_final = df_construccion[columnas_materiales].copy()

# 5. Quitar el sufijo '_materiales' para dejar solo las ciudades
df_materiales_final.columns = [
    col.replace('_materiales', '') for col in df_materiales_final.columns
]

correccion_columnas_materiales = {
    'juárez': 'cd. juárez',
    'cd. jiménez chih.': 'jiménez'
}

# Renombramos las columnas defectuosas en df_materiales_final si existen
df_materiales_final = df_materiales_final.rename(columns=correccion_columnas_materiales)

print("Estructura final del DataFrame:")
print(f"Filas: {df_materiales_final.shape[0]}, Columnas: {df_materiales_final.shape[1]}")

Estructura final del DataFrame:
Filas: 103, Columnas: 48


In [15]:
# =============================================================================
# 1. CONFIGURACIÓN DE PERÍODOS Y MAPEOS
# =============================================================================
# Diccionario para mapear la 'Class' (de df_precios_promedios) a la columna de INPC
from inpc_lista_productos import mapping_productos_inpc
print(f"Mapeo de productos INPC cargado: {mapping_productos_inpc}")

# Para df_materiales_final (INPP): Los textos exactos de la columna 'fecha_o_titulo'
FECHA_BASE_INPP = 'Jul 2018'
FECHAS_OBJETIVO_INPP = ['Ago 2022', 'Sep 2022', 'Oct 2022', 'Nov 2022']

# índice 6 para Jul 2018 (base), índices 55 al 58 para Ago-Nov 2022 (objetivo)
IDX_BASE_INPC = 6 
IDX_OBJETIVO_INPC = [55, 56, 57, 58]

Mapeo de productos INPC cargado: {'Tortilla de maíz': '014 Tortilla de maíz', 'Pan blanco': '008 Pan blanco', 'Pan dulce': '010 Pan dulce', 'Pollo': '022 Pollo', 'Carne de res': '018 Carne de res', 'Vísceras de res': '025 Vísceras de res', 'Chorizo': '020 Chorizo', 'Jamón': '021 Jamón', 'Salchichas': '023 Salchichas', 'Tocino': '024 Tocino', 'Leche pasteurizada': '034 Leche pasteurizada y fresca', 'Leche en polvo': '032 Leche en polvo', 'Leche evaporada': '033 Leche evaporada y condensada', 'Crema de leche': '030 Crema y otros productos a base de leche', 'Queso amarillo': '036 Queso amarillo', 'Queso fresco': '037 Queso fresco', 'Queso manchego o Chihuahua': '038 Queso manchego y Chihuahua', 'Queso Oaxaca o asadero': '039 Queso Oaxaca y asadero', 'Mantequilla': '044 Mantequilla', 'Huevo': '031 Huevo', 'Aguacate': '045 Aguacate', 'Guayaba': '047 Guayaba', 'Limón': '048 Limón', 'Manzana': '049 Manzana', 'Melón': '050 Melón', 'Naranja': '051 Naranja', 'Papaya': '052 Papaya', 'Piña': '054 

In [16]:
# Diccionario para mapear 'City_name' a la llave del diccionario ciudades_inpc
from inpc_lista_productos import mapping_ciudades_inpc
print(f"Mapeo de ciudades INPC cargado: {mapping_ciudades_inpc}")

Mapeo de ciudades INPC cargado: {'Acapulco, Gro.': 'inpc_acapulco', 'Aguascalientes, Ags.': 'inpc_aguascalientes', 'Campeche, Camp.': 'inpc_campeche', 'Área Metropolitana de la Cd. de México': 'inpc_cdmx', 'Cd. Acuña, Coah.': 'inpc_cd_acuna', 'Jiménez, Chih.': 'inpc_cd_jimenez', 'Cd. Juárez, Chih.': 'inpc_cd_juarez', 'Chetumal, Q.R.': 'inpc_chetumal', 'Chihuahua, Chih.': 'inpc_chihuahua', 'Colima, Col.': 'inpc_colima', 'Córdoba, Ver.': 'inpc_cordoba', 'Cortazar, Gto.': 'inpc_cortazar', 'Cuernavaca, Mor.': 'inpc_cuernavaca', 'Culiacán, Sin.': 'inpc_culiacan', 'Durango, Dgo.': 'inpc_durango', 'Fresnillo, Zac.': 'inpc_fresnillo', 'Guadalajara, Jal.': 'inpc_guadalajara', 'Hermosillo, Son.': 'inpc_hermosillo', 'Huatabampo, Son.': 'inpc_huatabampo', 'Iguala, Gro.': 'inpc_iguala', 'Jacona, Mich.': 'inpc_jacona', 'La Paz, B.C.S.': 'inpc_la_paz', 'León, Gto.': 'inpc_leon', 'Matamoros, Tamps.': 'inpc_matamoros', 'Mérida, Yuc.': 'inpc_merida', 'Mexicali, B.C.': 'inpc_mexicali', 'Monclova, Coah.':

In [17]:
print(df_precios_promedios)

       Year  Month                       Date  Id_city          City_name  \
0      2018      7  17/08/2018 12:00:00 a. m.       43    Campeche, Camp.   
1      2018      7  17/08/2018 12:00:00 a. m.       43    Campeche, Camp.   
2      2018      7  17/08/2018 12:00:00 a. m.       43    Campeche, Camp.   
3      2018      7  17/08/2018 12:00:00 a. m.       43    Campeche, Camp.   
4      2018      7  17/08/2018 12:00:00 a. m.       43    Campeche, Camp.   
...     ...    ...                        ...      ...                ...   
19856  2018      7  17/08/2018 12:00:00 a. m.        4  Guadalajara, Jal.   
19857  2018      7  17/08/2018 12:00:00 a. m.        4  Guadalajara, Jal.   
19858  2018      7  17/08/2018 12:00:00 a. m.        4  Guadalajara, Jal.   
19859  2018      7  17/08/2018 12:00:00 a. m.        4  Guadalajara, Jal.   
19860  2018      7  17/08/2018 12:00:00 a. m.        4  Guadalajara, Jal.   

                              Category              Sub_category  \
0      

In [18]:
# =============================================================================
# 2. FUNCIONES DE DEFLACTACIÓN (INPC e INPP)
# =============================================================================

def deflactar_inpc_2022(precio_base, ciudad, clase_producto):
    """Calcula el precio ajustado a 2022 usando los DataFrames de INPC."""
    # 1. Validar que tengamos el mapeo
    if ciudad not in mapping_ciudades_inpc or clase_producto not in mapping_productos_inpc:
        return np.nan
        
    inpc_key = mapping_ciudades_inpc[ciudad]
    col_inpc = mapping_productos_inpc[clase_producto]
    
    # 2. Extraer el DataFrame de la ciudad específica
    df_inpc = ciudades_inpc.get(inpc_key)
    if df_inpc is None or col_inpc not in df_inpc.columns:
        return np.nan
        
    # 3. Extraer valores (asegurar que sean numéricos)
    try:
        inpc_base_val = df_inpc.loc[IDX_BASE_INPC, col_inpc]
        inpc_periodo_vals = df_inpc.loc[IDX_OBJETIVO_INPC, col_inpc].astype(float)
    except KeyError:
        return np.nan # Si los índices no existen
        
    if inpc_base_val == 0 or pd.isna(inpc_base_val):
        return np.nan
        
    # 4. Cálculo matemático: P_base * mediana(INPC_obj / INPC_base)
    cocientes = inpc_periodo_vals / inpc_base_val
    return precio_base * np.median(cocientes)


def deflactar_inpp_materiales(precio_base, ciudad_columna):
    """Calcula el precio ajustado a 2022 usando df_materiales_final."""
    if ciudad_columna not in df_materiales_final.columns:
        return precio_base # Si no hay datos, retornamos el precio original (misma lógica original)
        
    # Extraer fila base y filas objetivo usando la columna 'fecha_o_titulo'
    fila_base = df_materiales_final[df_materiales_final['fecha_o_titulo'] == FECHA_BASE_INPP]
    filas_objetivo = df_materiales_final[df_materiales_final['fecha_o_titulo'].isin(FECHAS_OBJETIVO_INPP)]
    
    if fila_base.empty or filas_objetivo.empty:
        return precio_base
        
    inpp_base_val = float(fila_base.iloc[0][ciudad_columna])
    inpp_periodo_vals = filas_objetivo[ciudad_columna].astype(float)
    
    if inpp_base_val == 0 or pd.isna(inpp_base_val):
        return precio_base
        
    cocientes = inpp_periodo_vals / inpp_base_val
    return precio_base * np.median(cocientes)


In [19]:
# =============================================================================
# 3. APLICACIÓN AL DATAFRAME DE PRECIOS PROMEDIOS
# =============================================================================

# Limpiamos NaN en precios por seguridad
df_precios_promedios = df_precios_promedios.dropna(subset=['Price'])

# Aplicamos la función fila por fila
# Usamos una función lambda para pasar los valores de cada fila a nuestra función deflactora
df_precios_promedios['Price_2022'] = df_precios_promedios.apply(
    lambda row: deflactar_inpc_2022(row['Price'], row['City_name'], row['Class']),
    axis=1
)

# --- TRATAMIENTO ESPECIAL PARA MATERIALES DE CONSTRUCCIÓN ---
# Como los materiales no están en df_precios_promedios, los inicializamos en 100
# calculamos su ajuste y los agregamos como nuevas filas al DataFrame final.

registros_materiales = []

# Iteramos sobre todas las ciudades que ya definimos en tu diccionario mapping_ciudades_inpc
for ciudad in mapping_ciudades_inpc.keys():
    
    # 1. Limpiamos el nombre para que coincida con las columnas de df_materiales_final
    ciudad_columna = ciudad.split(',')[0].strip().lower()
    
    # 2. Calculamos el precio ajustado partiendo de un precio base de 100
    precio_base_materiales = 100.0
    precio_ajustado = deflactar_inpp_materiales(precio_base_materiales, ciudad_columna)
    
    # 3. Creamos un nuevo registro estructurado con las mismas columnas que df_precios_promedios
    registros_materiales.append({
        'Year': 2018,       
        'Month': 7,        
        'Date': '17/08/2018 12:00:00 a. m.',      
        #'Id_city',  
        'City_name': ciudad,  
        'Category': '9. Construcción',  
        'Sub_category': '9.1. Materiales',  
        'Group': '9.1.1. Materiales de construcción',  
        'Sub_group': '91 Materiales de construcción',  
        #'Id_class',  
        'Class': 'Materiales de construcción',  
        #'Id_product', 
        'Product_name': 'Índice INPP para los materiales de construccion (Base 100)', 
        'Price': precio_base_materiales,  
        'Quantity': 1,  
        'Unit': '',  
        'No_data': '',
        'Price_2022': precio_ajustado
    })

# 4. Convertimos la lista de nuevos registros a un DataFrame
df_nuevos_materiales = pd.DataFrame(registros_materiales)

# 5. Concatenamos estos nuevos registros al final del DataFrame principal
df_precios_promedios = pd.concat([df_precios_promedios, df_nuevos_materiales], ignore_index=True)

print(f"Se agregaron {len(df_nuevos_materiales)} registros de materiales de construcción.")


Se agregaron 46 registros de materiales de construcción.


In [20]:
print("Muestra de precios promedios ajustados a 2022:")
print(df_precios_promedios[['City_name', 'Class', 'Product_name', 'Price', 'Price_2022']].head(10))

Muestra de precios promedios ajustados a 2022:
         City_name                   Class  \
0  Campeche, Camp.            Queso fresco   
1  Campeche, Camp.            Queso fresco   
2  Campeche, Camp.            Queso fresco   
3  Campeche, Camp.            Queso fresco   
4  Campeche, Camp.            Otros quesos   
5  Campeche, Camp.            Otros quesos   
6  Campeche, Camp.            Otros quesos   
7  Campeche, Camp.            Otros quesos   
8  Campeche, Camp.  Queso Oaxaca o asadero   
9  Campeche, Camp.  Queso Oaxaca o asadero   

                                      Product_name   Price  Price_2022  
0  KRAFT, DOBLE CREMA, PHILADELPHIA, PAQ DE 190 GR  147.37  199.932442  
1                        GONELA, COTTAGE, A GRANEL  103.50  140.415334  
2                      EL CIERVO, PANELA, A GRANEL  132.80  180.165762  
3                                 PANELA, A GRANEL  108.25  146.859516  
4                 PARMA, PARMESANO, BOTE DE 227 GR  328.19         NaN  
5       

## 2. Carga y preparación de microdatos ENIGH 2022

### Sección 2 — Microdatos ENIGH 2022

**Archivos utilizados:**
- `concentradohogar.csv` (90,102 rows x 126): Un registro por hogar.
  Variables clave: folio (folioviv o foliohog?), tam_loc /, factor_hog (factor ?), clase_hog / , sexo_jefe /, edad_jefe /,
  educa_jefe /, tot_integ /, menores /, ing_total (ing_cor ?), gasto_mon /, mater_serv /, entidad_fed X,
  clave_municipio X.
- `gastoshogar.csv`: Gastos monetarios a nivel producto-hogar.
- `gastospersona.csv`: Gastos individuales (ropa, calzado, salud, educación).
  Se suma al gasto de hogar para las categorías relevantes.
- `viviendas.csv`: Situación de tenencia. **Corrección clave:**
  códigos 3 (propia pagándose) y 4 (propia pagada) = vivienda propia.
  El programa de 2006 usaba códigos 4 y 5 — diferencia que genera ~1,757 hogares extra.
- `hogares.csv`: Para Z9 (AUTOLAV).
- `datos_municipios_latitud_longitud.asc` (304,568 × 4): Para asignar cada hogar a su
  ciudad de referencia (distancia geodésica, límite 400 km). --> ese se queda

**Filtros de muestra (idénticos al Gauss 2014):**
1. Vivienda propia (tenencia = 3 ó 4)
2. clase_hog ≤ 5 (excluye hogares en viviendas colectivas)
3. Edad del jefe: 20–75 años
4. Integrantes totales ≤ 8
5. Gasto monetario ≥ percentil 0.1%
6. Distancia a ciudad INPC más cercana ≤ 400 km

**Muestra resultante:** 12,592 hogares tras filtros, 12,372 tras filtro de categorías
con gasto ≥ $10, 8,940 tras el trim iterativo del 1% × 16 iteraciones.

**Asignación de precios:** Cada hogar recibe los precios de la ciudad INPC más cercana
según distancia del gran círculo (fórmula esférica).


In [21]:
# ---------------------------------------------------------------
# 2.1 Municipios con latitud/longitud 
# ---------------------------------------------------------------
from municipios import municipios_objetivo_46

print('Cargando municipios lat/lon...')
municipios = pd.read_csv(DATA_DIR + 'datos_geograficos_municipios.csv', dtype={'CVEGEO': str})

# 1. Crear columnas temporales directamente en el dataframe original
municipios['NOM_MUN_CLEAN'] = municipios['NOM_MUN'].str.lower().str.strip()
municipios['CVE_LOC_NUM'] = pd.to_numeric(municipios['CVE_LOC'], errors='coerce')

# 2. Aplicar AMBOS filtros al mismo tiempo y asegurar la copia independiente
condicion_municipio = municipios['NOM_MUN_CLEAN'].isin(municipios_objetivo_46)
condicion_cabecera = municipios['CVE_LOC_NUM'] == 1

municipios = municipios[condicion_municipio & condicion_cabecera].copy()

municipios['ubica_geo'] = municipios['CVEGEO'].str[:4]

# 3. Limpieza final de columnas temporales
municipios = municipios.drop(columns=['NOM_MUN_CLEAN', 'CVE_LOC_NUM'])

print(f'Municipios shape final: {municipios.shape}')


# ---------------------------------------------------------------
# 2.2 Gastos hogar ENIGH 2022
#     Columnas: folioviv, clave_gasto_numerica, gasto_tri, ...(10 cols total)
# ---------------------------------------------------------------
print('Cargando gastos hogar 2022...')
gastos_hogares = pd.read_csv(DATA_DIR + 'gastoshogar.csv')
print(f'  Gastos hogar shape: {gastos_hogares.shape}')

# ---------------------------------------------------------------
# 2.3 Gastos persona ENIGH 2022
# ---------------------------------------------------------------
print('Cargando gastos persona 2022...')
gastos_persona = pd.read_csv(DATA_DIR + 'gastospersona.csv')
print(f'  Gastos persona shape: {gastos_persona.shape}')

# ---------------------------------------------------------------
# 2.4 Concentrado hogares 
# ---------------------------------------------------------------
print('Cargando concentrado hogares 2022...')
conc = pd.read_csv(DATA_DIR + 'concentradohogar.csv')
print(f'  Concentrado shape: {conc.shape}')

# ---------------------------------------------------------------
# 2.4 Concentrado viviendas 
# ---------------------------------------------------------------
print('Cargando viviendas 2022...')
viviendas = pd.read_csv(DATA_DIR + 'viviendas.csv')
print(f'  Viviendas shape: {viviendas.shape}')

# ---------------------------------------------------------------
# 2.4 Hogares 
# ---------------------------------------------------------------
print('Cargando hogares 2022...')
hogares = pd.read_csv(DATA_DIR + 'hogares.csv')
print(f'  Hogares shape: {hogares.shape}')

Cargando municipios lat/lon...
Municipios shape final: (58, 21)
Cargando gastos hogar 2022...
  Gastos hogar shape: (5075174, 31)
Cargando gastos persona 2022...
  Gastos persona shape: (402557, 24)
Cargando concentrado hogares 2022...
  Concentrado shape: (90102, 126)
Cargando viviendas 2022...
  Viviendas shape: (88823, 64)
Cargando hogares 2022...
  Hogares shape: (90102, 141)


In [22]:
num_hogares = len(conc['folioviv']) 
print(f'Hogares totales antes de filtros: {num_hogares}')

# ---------------------------------------------------------------
# 2.7 Vivienda propia
#
# CORRECCIÓN v3: el programa Gauss 2014 usa tenencia == 3 OR 4
#   (propia pagándose = 3, propia totalmente pagada = 4)
# El código de 2006 usaba 4 OR 5 — ese era el bug en v2.
# ---------------------------------------------------------------

estatus_ten = (viviendas.set_index('folioviv')['tenencia']
               .isin([3, 4])
               .astype(float))

vivienda_propia = np.array([estatus_ten.get(fol, 0.0)
                             for fol in conc['folioviv']])



# ---------------------------------------------------------------
# 2.8 Filtro de muestra (idéntico al Gauss 2014, línea 1101)
#   vivienda_propia > 0
#   clase_hog <= 5
#   edad_jefe 20-75
#   integrantes <= 8
#   gasto_mon >= percentil 0.1%
# ---------------------------------------------------------------
p001 = np.quantile(conc['gasto_mon'], 0.001)
mask_filtro = (
    (vivienda_propia > 0) &
    (conc['clase_hog']  <= 5) & #inecesario porque todos son inferiores a 5
    (conc['edad_jefe']   >= 20) &
    (conc['edad_jefe']  <= 75) &
    (conc['tot_integ'] <= 8) &
    (conc['gasto_mon'] >= p001)
)
conc = conc[mask_filtro]
num_hogares = len(conc['folioviv'])
print(f'Hogares después de filtros básicos: {num_hogares}')


Hogares totales antes de filtros: 90102
Hogares después de filtros básicos: 57989


In [23]:
# ---------------------------------------------------------------
# PREPARACIÓN DE DATOS GEOGRÁFICOS (PANDAS)
# ---------------------------------------------------------------
print('Cargando catálogo completo de municipios...')
catalogo_completo_municipios = pd.read_csv(DATA_DIR + 'datos_geograficos_municipios.csv', dtype={'CVEGEO': str})

# Normalizar y crear ubica_geo
catalogo_completo_municipios['NOM_MUN_CLEAN'] = catalogo_completo_municipios['NOM_MUN'].str.lower().str.strip()
catalogo_completo_municipios['CVE_LOC_NUM']   = pd.to_numeric(catalogo_completo_municipios['CVE_LOC'], errors='coerce')
catalogo_completo_municipios['ubica_geo']     = catalogo_completo_municipios['CVEGEO'].str[:5]

# Filtro 1: Solo cabeceras municipales para tener un punto central por municipio
catalogo_completo_municipios = catalogo_completo_municipios[catalogo_completo_municipios['CVE_LOC_NUM'] == 1].copy()

# Crear subconjunto de las 46 ciudades objetivo
ciudades_inpc = catalogo_completo_municipios[catalogo_completo_municipios['NOM_MUN_CLEAN'].isin(municipios_objetivo_46)].copy()

# ---------------------------------------------------------------
# 2.9 Asignar lat/lon a cada hogar usando ubica_geo (¡Estilo Pandas!)
# ---------------------------------------------------------------
# Convertimos la columna a string y aseguramos los 4 dígitos vectorialmente
ubica_geo_hogar = conc['ubica_geo'].astype(int).astype(str).str.zfill(5).values

# Crear lookup dict desde el catálogo COMPLETO: ubica_geo -> (lat, lon)
lookup_geo = dict(zip(catalogo_completo_municipios['ubica_geo'], 
                      zip(catalogo_completo_municipios['LAT_DECIMAL'], catalogo_completo_municipios['LON_DECIMAL'])))
lookup_nombre = dict(zip(catalogo_completo_municipios['ubica_geo'], catalogo_completo_municipios['NOM_MUN']))

print(f"1. Hogares iniciales en 'conc': {len(conc)}")

# Mapeamos el vector usando el diccionario de forma directa
coordenadas = [lookup_geo.get(geo, (np.nan, np.nan)) for geo in ubica_geo_hogar]
latitud_hogar, longitud_hogar = zip(*coordenadas)

# Extraer el nombre del municipio de la vivienda (asignamos 'Desconocido' si no cruza)
nombre_municipio_origen = np.array([lookup_nombre.get(geo, 'Desconocido') for geo in ubica_geo_hogar])

latitud_hogar = np.array(latitud_hogar)
longitud_hogar = np.array(longitud_hogar)

# Limpiar hogares que no cruzaron en el diccionario
hogares_validos = ~np.isnan(latitud_hogar)
conc            = conc[hogares_validos].copy() 
latitud_hogar   = latitud_hogar[hogares_validos]
longitud_hogar  = longitud_hogar[hogares_validos]
nombre_municipio_origen = nombre_municipio_origen[hogares_validos]

# ---------------------------------------------------------------
# 2.10 Validación y Encontrar ciudad INPC más cercana
# ---------------------------------------------------------------
# 1. Extraer coordenadas de las ciudades
lat_ciudades_rad = np.radians(ciudades_inpc['LAT_DECIMAL'].values)
lon_ciudades_rad = np.radians(ciudades_inpc['LON_DECIMAL'].values)

# 2. Pasar a radianes con expansión de dimensiones explícita
lat_hogar_rad = np.radians(latitud_hogar).reshape(-1, 1)
lon_hogar_rad = np.radians(longitud_hogar).reshape(-1, 1)

# 3. Fórmula esférica vectorizada
arg = (np.sin(lat_hogar_rad) * np.sin(lat_ciudades_rad) +
       np.cos(lat_hogar_rad) * np.cos(lat_ciudades_rad) *
       np.cos(lon_ciudades_rad - lon_hogar_rad))

arg = np.clip(arg, -1.0, 1.0)
matriz_distancias = np.arccos(arg) * 6371.0

# 4. Obtener índice y distancia mínima
idx_ciudad_mas_cercana  = np.argmin(matriz_distancias, axis=1)
dist_ciudad_mas_cercana = np.min(matriz_distancias, axis=1)

# ---------------------------------------------------------------
# 2.11 Filtrar a <= 400 km y concatenar nuevas columnas a 'conc'
# ---------------------------------------------------------------
distancia_maxima = 400.0
mask_dist = dist_ciudad_mas_cercana <= distancia_maxima

# Aplicar máscara a los vectores
conc                    = conc[mask_dist].copy() # Aseguramos que conc sea una copia independiente
latitud_hogar_filtrada  = latitud_hogar[mask_dist]
longitud_hogar_filtrada = longitud_hogar[mask_dist]
idx_ciudad_filtrada     = idx_ciudad_mas_cercana[mask_dist]
nombre_municipio_filtrado = nombre_municipio_origen[mask_dist]

#  Extraer los nombres de los municipios desde el DataFrame de ciudades INPC
# Usamos 'NOM_MUN' para tener el nombre original con mayúsculas y acentos
nombres_ciudades_array = ciudades_inpc['NOM_MUN'].values

# Mapear los índices a sus respectivos nombres de forma vectorizada
nombres_asignados = nombres_ciudades_array[idx_ciudad_filtrada]

# Asignar todas las columnas nuevas al DataFrame 'conc'
conc['nombre_municipio_hogar'] = nombre_municipio_filtrado
conc['latitud_hogar']         = latitud_hogar_filtrada
conc['longitud_hogar']        = longitud_hogar_filtrada
#conc['idx_ciudad_cercana']    = idx_ciudad_filtrada
conc['nombre_ciudad_cercana'] = nombres_asignados 

print(f'Hogares finales (<=400 km): {len(conc)}')
print(conc[['ubica_geo', 'latitud_hogar', 'longitud_hogar', 'nombre_ciudad_cercana']].head())

Cargando catálogo completo de municipios...
1. Hogares iniciales en 'conc': 57989
Hogares finales (<=400 km): 57989
   ubica_geo  latitud_hogar  longitud_hogar nombre_ciudad_cercana
2       1001      21.879822     -102.296046        Aguascalientes
4       1001      21.879822     -102.296046        Aguascalientes
5       1001      21.879822     -102.296046        Aguascalientes
7       1001      21.879822     -102.296046        Aguascalientes
8       1001      21.879822     -102.296046        Aguascalientes


In [24]:
print(f"Columnas de conc tras asignación de lat/lon y ciudad cercana:{list(conc.columns)}")

Columnas de conc tras asignación de lat/lon y ciudad cercana:['folioviv', 'foliohog', 'ubica_geo', 'tam_loc', 'est_socio', 'est_dis', 'upm', 'factor', 'clase_hog', 'sexo_jefe', 'edad_jefe', 'educa_jefe', 'tot_integ', 'hombres', 'mujeres', 'mayores', 'menores', 'p12_64', 'p65mas', 'ocupados', 'percep_ing', 'perc_ocupa', 'ing_cor', 'ingtrab', 'trabajo', 'sueldos', 'horas_extr', 'comisiones', 'aguinaldo', 'indemtrab', 'otra_rem', 'remu_espec', 'negocio', 'noagrop', 'industria', 'comercio', 'servicios', 'agrope', 'agricolas', 'pecuarios', 'reproducc', 'pesca', 'otros_trab', 'rentas', 'utilidad', 'arrenda', 'transfer', 'jubilacion', 'becas', 'donativos', 'remesas', 'bene_gob', 'transf_hog', 'trans_inst', 'estim_alqu', 'otros_ing', 'gasto_mon', 'alimentos', 'ali_dentro', 'cereales', 'carnes', 'pescado', 'leche', 'huevo', 'aceites', 'tuberculo', 'verduras', 'frutas', 'azucar', 'cafe', 'especias', 'otros_alim', 'bebidas', 'ali_fuera', 'tabaco', 'vesti_calz', 'vestido', 'calzado', 'vivienda', '

## 3. Construcción de gastos y categorías de demanda

### Sección 3 — Categorías de gasto y variables del modelo

**12 categorías de gasto** (Cuadro 1 del paper):

| Cat | Nombre | Subproductos ENIGH | Claves |
|-----|--------|--------------------|--------|
| 1 | Tortillas de maíz | 1 | A004 |
| 2 | Pan | 2 | A012, A013-A014 |
| 3 | Pollo y huevo | 3 | A057-A058, A059, A093 |
| 4 | Carne de res | 3 | A025, A034, A037 |
| 5 | Carnes procesadas | 4 | A049, A052, A055, A054 |
| 6 | Bebidas no alcohólicas | 3 | A218, A220, A215 |
| 7 | Frutas | 11 | A158, A065-A067, A161, ... |
| 8 | Verduras | 17 | A108, A124, A102, ... |
| 9 | Lácteos | 9 | A075, A078, A079, A076, ... |
| 10 | Materiales de construcción | 1 | K044 |
| 11 | Transporte foráneo | 2 | M001, M003 |
| 12 | Medicamentos | 8 grupos | J028+J052, J031+J056, ... |

**Nota sobre el orden:** El paper usa el orden [1-Tortillas, 2-Pan, ..., 10-Materiales,
11-Transporte, 12-Medicamentos]. El notebook replica exactamente este orden con
Materiales como numéraire (categoría 12) en la estimación con simetría.

**Índice de precios Divisia por categoría** (Lewbel 1989, Ecuación 2 del paper):
$$\mathcal{P}_{jh} = \frac{1}{k_j} \prod_{i=1}^{n_j} \left(\frac{p_{ji}}{w_{jih}}\right)^{w_{jih}}$$
donde $k_j = \prod_i \bar{w}_{ji}^{-\bar{w}_{ji}}$ y $\bar{w}_{ji}$ es el share promedio
muestral del subproducto *i* en la categoría *j*.

**Variables Z (características del hogar):**
- Z1: EDUC — educación del jefe (años)
- Z2: INTEGRANTES — total integrantes
- Z3: EDUCxINTEGRANTES
- Z4: MENORES — integrantes < 12 años
- Z5: INGR80 — indicadora ingreso > decil 8 (col 22 del concentrado)
- Z6: EDUCxMENORES
- Z7: EDUC²
- Z8: LOC2500 — indicadora localidad < 2,500 hab (tam_loc = 4)
- Z9: AUTOLAV — indicadora posee auto Y lavadora


In [25]:
# Lista de claves numéricas de productos de interés (57 productos)
from productos import CLAVES
print(f"Numero de productos de interés: {len(CLAVES)}")

Numero de productos de interés: 57


In [26]:
# Categorías de demanda de productos
from productos import CATEGORIAS
print(f"Categorías de productos: {CATEGORIAS}")

Categorías de productos: {'Tortillas': ['Tortilla de maíz'], 'Pan': ['Pan blanco', 'Pan dulce'], 'Pollo y huevo': ['Pollo', 'Huevo'], 'Carne de res': ['Carne de res', 'Vísceras de res'], 'Carnes procesadas': ['Chorizo', 'Jamón', 'Salchichas', 'Tocino'], 'Lácteos': ['Leche pasteurizada y fresca', 'Leche en polvo', 'Leche evaporada, condensada y maternizada', 'Crema de leche', 'Queso amarillo', 'Queso fresco', 'Queso manchego o Chihuahua', 'Queso Oaxaca o asadero', 'Mantequilla'], 'Frutas': ['Aguacate', 'Guayaba', 'Limón', 'Manzana', 'Melón', 'Naranja', 'Papaya', 'Piña', 'Plátanos', 'Sandía', 'Uva'], 'Verduras': ['Calabacita', 'Cebolla', 'Chayote', 'Chile poblano', 'Chile serrano', 'Ejotes', 'Jitomate', 'Lechuga y col', 'Nopales', 'Papa y otros tubérculos', 'Pepino', 'Tomate verde', 'Zanahoria', 'Frijol'], 'Bebidas': ['Jugos o néctares envasados', 'Agua embotellada', 'Refrescos envasados'], 'Medicamentos': ['Analgésicos', 'Antibióticos', 'Antigripales', 'Cardiovasculares', 'Dermatológico

In [27]:
# ==========================================
# 1. DICCIONARIOS Y MAPEOS
# ==========================================

codigo_a_producto = {codigo: producto for producto, codigos in CLAVES.items() for codigo in codigos}
producto_a_categoria = {producto: cat for cat, productos in CATEGORIAS.items() for producto in productos}
codigo_a_categoria = {codigo: producto_a_categoria[producto] for codigo, producto in codigo_a_producto.items()}

# ==========================================
# 2. FILTRO Y PIVOT DE GASTOS
# ==========================================
claves_validas = list(codigo_a_producto.keys())
gastos_hogares = gastos_hogares[gastos_hogares['clave'].isin(claves_validas)]
gastos_persona = gastos_persona[gastos_persona['clave'].isin(claves_validas)]

gastos_totales = pd.concat([gastos_hogares[['folioviv', 'foliohog', 'clave', 'gasto_tri']], 
                            gastos_persona[['folioviv', 'foliohog', 'clave', 'gasto_tri']]])

gastos_totales['gasto_tri'] = pd.to_numeric(gastos_totales['gasto_tri'], errors='coerce')
gastos_totales['producto'] = gastos_totales['clave'].map(codigo_a_producto)
gastos_totales['categoria'] = gastos_totales['clave'].map(codigo_a_categoria)

# Unificar tipos para que el índice coincida después
gastos_totales[['folioviv', 'foliohog']] = gastos_totales[['folioviv', 'foliohog']].astype(str)
conc[['folioviv', 'foliohog']] = conc[['folioviv', 'foliohog']].astype(str)

gastos_por_producto = gastos_totales.pivot_table(index=['folioviv', 'foliohog'], columns='producto', values='gasto_tri', aggfunc='sum').fillna(0)
gastos_por_categoria = gastos_totales.pivot_table(index=['folioviv', 'foliohog'], columns='categoria', values='gasto_tri', aggfunc='sum').fillna(0)

# ==========================================
# 3. LIMPIEZA, FILTRO Y CRUCE DE PRECIOS
# ==========================================
def normalizar_ciudad(texto):
    if not isinstance(texto, str): return ""
    texto_limpio = texto.split(',')[0].strip().lower()
    return "".join(c for c in unicodedata.normalize('NFD', texto_limpio) if unicodedata.category(c) != 'Mn')

# ERROR 1 CORREGIDO: Filtrar ANTES de agrupar
df_precios_promedios = df_precios_promedios.dropna(subset=['Price_2022']).copy()

# Precios
precios_ciudad = df_precios_promedios.groupby(['City_name', 'Class'])['Price_2022'].mean().reset_index()
precios_ciudad['ciudad_match'] = precios_ciudad['City_name'].apply(normalizar_ciudad)

# Hogares
hogares_ciudades = conc[['folioviv', 'foliohog', 'nombre_ciudad_cercana']].copy()
hogares_ciudades['ciudad_match'] = hogares_ciudades['nombre_ciudad_cercana'].apply(normalizar_ciudad)

homologaciones = {
    'othon p blanco': 'chetumal', 'centro': 'villahermosa', 
    'cuauhtemoc': 'ciudad de mexico', 'huatabampo': 'hermosillo'
}
hogares_ciudades['ciudad_match'] = hogares_ciudades['ciudad_match'].replace(homologaciones)


join_precios_hogar = hogares_ciudades.merge(precios_ciudad, on='ciudad_match', how='left')

# ==========================================
# 4. PIVOT DE PRECIOS Y ALINEACIÓN DE ÍNDICES
# ==========================================
precios_wide = join_precios_hogar.pivot_table(index=['folioviv', 'foliohog'], columns='Class', values='Price_2022', aggfunc='mean').fillna(0)

hogares_index = gastos_por_producto.index
precios_wide = precios_wide.reindex(hogares_index).fillna(0)
gastos_por_categoria = gastos_por_categoria.reindex(hogares_index).fillna(0)

# ==========================================
# 5. CÁLCULO DE DIVISIA (OPTIMIZADO)
# ==========================================
def divisia_price_index(gastos_componentes, precios_componentes, gasto_total_cat):
    n_prod = len(gastos_componentes)
    N = len(gasto_total_cat)
    w = np.zeros((n_prod, N))
    gasto_cat_valido = np.where(gasto_total_cat > 0, gasto_total_cat, 1e-10)
    
    for j in range(n_prod): w[j] = gastos_componentes[j] / gasto_cat_valido
    w_bar = w.mean(axis=1)
    w_bar_seguro = np.where(w_bar > 0, w_bar, 1e-10)
    log_k = -np.sum(w_bar * np.log(w_bar_seguro))
    k = np.exp(log_k)
    
    log_P = np.zeros(N)
    for j in range(n_prod):
        wj = np.where(w[j] > 0, w[j], 1e-10)
        Pj = np.where(precios_componentes[j] > 0, precios_componentes[j], 1e-10)
        log_P += w[j] * np.log(Pj / wj)
    return np.exp(log_P - log_k)

def get_vec(df, col, N):
    return df[col].values if col in df.columns else np.zeros(N)

N_hogares = len(hogares_index)
indices_finales = {}

# ¡MAGIA!: Este for reemplaza las casi 60 líneas donde declarabas categoría por categoría
for cat_nombre, lista_productos in CATEGORIAS.items():
    if len(lista_productos) == 1:
        # Si la categoría solo tiene 1 producto (ej. Tortillas), tomamos su precio directo
        indices_finales[cat_nombre] = get_vec(precios_wide, lista_productos[0], N_hogares)
    else:
        # Si tiene más de 1, calculamos Divisia iterando sobre sus productos
        indices_finales[cat_nombre] = divisia_price_index(
            [get_vec(gastos_por_producto, p, N_hogares) for p in lista_productos],
            [get_vec(precios_wide, p, N_hogares) for p in lista_productos],
            get_vec(gastos_por_categoria, cat_nombre, N_hogares)
        )

# Construir el DataFrame final
df_precios_divisia = pd.DataFrame(indices_finales, index=hogares_index).reset_index()
print("Cálculo finalizado con éxito.")

Cálculo finalizado con éxito.


In [28]:
print(df_precios_divisia.describe())

          Tortillas           Pan  Pollo y huevo  Carne de res  \
count  89515.000000  8.951500e+04   8.951500e+04  8.951500e+04   
mean       9.419716  7.594205e-01   1.396178e+01  2.177573e+01   
std       10.822798  1.344075e+00   2.121804e+01  4.740196e+01   
min        0.000000  5.120173e-11   4.822074e-11  6.668585e-11   
25%        0.000000  5.120173e-11   6.986589e-11  6.668585e-01   
50%        0.000000  5.120173e-01   4.822074e-01  6.668585e-01   
75%       21.567033  5.120173e-01   2.858179e+01  6.668585e-01   
max       27.828955  9.008089e+00   8.486335e+01  2.474957e+02   

       Carnes procesadas       Lácteos        Frutas      Verduras  \
count       8.951500e+04  8.951500e+04  8.951500e+04  8.951500e+04   
mean        1.365360e+01  4.361314e+00  5.035762e+00  6.050965e+00   
std         3.496880e+01  1.633139e+01  1.057692e+01  9.127352e+00   
min         4.602964e-11  3.017313e-11  2.491142e-11  1.485862e-11   
25%         4.602964e-01  3.017313e-11  1.172562e-10  5

In [29]:
# ===============================================================
# 1. PREPARACIÓN DE ÍNDICES
# Aseguramos que todas las tablas usen el mismo índice doble
# ===============================================================
if 'folioviv' in df_precios_divisia.columns:
    df_precios_divisia = df_precios_divisia.set_index(['folioviv', 'foliohog'])

if 'folioviv' in conc.columns:
    conc = conc.set_index(['folioviv', 'foliohog'])

hogares_comunes = gastos_por_categoria.index.intersection(df_precios_divisia.index).intersection(conc.index)

gastos_por_categoria = gastos_por_categoria.loc[hogares_comunes]
df_precios_divisia = df_precios_divisia.loc[hogares_comunes]
conc = conc.loc[hogares_comunes]

print(f'Hogares comunes antes del filtro de gasto: {len(hogares_comunes)}')

# ===============================================================
# 2. CREACIÓN DE LA MÁSCARA (FILTRO)
# Hogares con al menos 1 categoría con gasto >= 10
# ===============================================================
# Cuenta cuántas columnas cumplen la condición por fila
categorias_relevantes = (gastos_por_categoria >= 10).sum(axis=1)
mask_categ = categorias_relevantes >= 1

# ===============================================================
# 3. APLICAR FILTRO A TODO
# ===============================================================
# .loc alineará automáticamente los índices de mask_categ con los de cada DF
gastos_por_categoria = gastos_por_categoria[mask_categ]
df_precios_divisia = df_precios_divisia[mask_categ]
conc = conc[mask_categ]

print(f'Hogares finales tras filtro de >= 10 pesos: {len(gastos_por_categoria)}')

# ===============================================================
# 4. PROPORCIONES DE GASTO (BUDGET SHARES)
# ===============================================================
# Suma el gasto total por hogar (suma por filas)
suma_gastos = gastos_por_categoria.sum(axis=1)

# Divide cada valor de gasto entre el gasto total del hogar
# .div(axis=0) asegura que la división se haga fila por fila
w_matrix = gastos_por_categoria.div(suma_gastos, axis=0)

# ===============================================================
# 5. PRECIOS EN LOGARITMOS
# ===============================================================
# Como acordamos, aplicamos np.log directo. Si llegara a existir un 0, 
# NumPy arrojará un "RuntimeWarning: divide by zero" y pondrá -inf.
precios_matrix_ln = np.log(df_precios_divisia)

Hogares comunes antes del filtro de gasto: 57721
Hogares finales tras filtro de >= 10 pesos: 57507


In [30]:
# ===============================================================
# 1. ALINEAR TABLA HOGARES
# Aseguramos que 'hogares' tenga el mismo índice y tamaño
# ===============================================================
hogares['folioviv'] = hogares['folioviv'].astype(str)
hogares['foliohog'] = hogares['foliohog'].astype(str)

if 'folioviv' in hogares.columns:
    hogares = hogares.set_index(['folioviv', 'foliohog'])


# Nos quedamos solo con los hogares que sobrevivieron al filtro de >= 10 pesos
hogares = hogares.loc[conc.index]

# ===============================================================
# 2. CONSTRUCCIÓN DE VARIABLES Z (Vectorizado)
# Usamos pd.to_numeric por si ENIGH cargó los números como texto
# ===============================================================

# Z1: Educación del jefe
Z1 = pd.to_numeric(conc['educa_jefe'], errors='coerce').fillna(0)

# Z2: Total de integrantes
Z2 = pd.to_numeric(conc['tot_integ'], errors='coerce')

# Z3: Interacción Educación x Integrantes
Z3 = Z1 * Z2

# Z4: Menores en el hogar
Z4 = pd.to_numeric(conc['menores'], errors='coerce')

# Z5: Ingreso en el top 20% (INGR80) usando 'ing_cor' de conc
ingreso_corriente = pd.to_numeric(conc['ing_cor'], errors='coerce')
p80 = ingreso_corriente.quantile(0.8)
Z5 = (ingreso_corriente >= p80).astype(float)

# Z6: Interacción Educación x Menores
Z6 = Z1 * Z4

# Z7: Educación al cuadrado
Z7 = Z1 ** 2

# Z8: Localidad Rural/Pequeña (LOC2500)
# tam_loc == 4 o '4' dependiendo de si es numérico o string
Z8 = (conc['tam_loc'].astype(str) == '4').astype(float)

# Z9: Tiene auto y lavadora
# Convertimos las columnas de hogares a booleanos (>0) y luego a float
tiene_lavadora = (pd.to_numeric(hogares['num_lavad'], errors='coerce') > 0).astype(float)
tiene_vehiculo = (pd.to_numeric(hogares['num_auto'], errors='coerce') > 0).astype(float)
Z9 = tiene_vehiculo * tiene_lavadora

# ===============================================================
# 3. FACTOR DE EXPANSIÓN Y EMPAQUETADO FINAL
# ===============================================================
# Factor para futuras agregaciones poblacionales
factor_hog = pd.to_numeric(conc['factor'], errors='coerce')

# Guardamos todo en un DataFrame limpio, conservando los folios como índice
Z_vars = pd.DataFrame({
    'Z1_educa_jefe': Z1,
    'Z2_tot_integ': Z2,
    'Z3_educXinteg': Z3,
    'Z4_menores': Z4,
    'Z5_ingr80': Z5,
    'Z6_educXmenores': Z6,
    'Z7_educ_cuad': Z7,
    'Z8_loc2500': Z8,
    'Z9_autolav': Z9
}, index=conc.index)

print('Variables Z construidas con éxito.')
print(f"  Media Z5 (INGR80):  {Z_vars['Z5_ingr80'].mean():.3f}  (esperado ≈ 0.200)")
print(f"  Media Z8 (LOC2500): {Z_vars['Z8_loc2500'].mean():.3f}")
print(f"  Media Z9 (AUTOLAV): {Z_vars['Z9_autolav'].mean():.3f}")

Variables Z construidas con éxito.
  Media Z5 (INGR80):  0.200  (esperado ≈ 0.200)
  Media Z8 (LOC2500): 0.414
  Media Z9 (AUTOLAV): 0.317


## 4. Estimación del sistema aproximado de demanda EASI

El sistema aproximado (ecuación 10 del paper) es lineal en parámetros:
$$w_{hj} = \sum_{r=0}^{3} b_r^j \tilde{y}_h^r + C^j z_h + \sum_{\ell} z_{\ell h} A_\ell^j p_h + (D^j z_h + B^j p_h) \tilde{y}_h + \varepsilon_{hj}$$

donde $\tilde{y}_h = \ln x_h - p_h' \bar{w}$ es la utilidad aproximada.

Se estiman las 11 ecuaciones (la 12 se deriva por aditividad) imponiendo simetría de las matrices $B$ y $A_\ell$, en 16 iteraciones que actualizan $\tilde{y}_h$.

### Sección 4 — Sistema aproximado de demanda EASI (Primera etapa)

**Modelo EASI** (Lewbel & Pendakur 2009, Ecuación 8 del paper):
$$\mathbf{w}_h = \sum_{r=0}^{3} \mathbf{b}_r \tilde{y}_h^r + \mathbf{C}z_h +
\mathbf{D}z_h \tilde{y}_h + \sum_{\ell=0}^{L} z_{\ell h} A_\ell \mathbf{p}_h +
\mathbf{B}\mathbf{p}_h \tilde{y}_h + \boldsymbol{\varepsilon}_h$$

donde $\tilde{y}_h = x_h - \mathbf{p}_h' \bar{\mathbf{w}}$ es la utilidad aproximada
y $\bar{\mathbf{w}}$ son las proporciones promedio de gasto.

**Parámetros a estimar:** 902 en total.
- **B** (12×12, simétrica): interacciones precio-precio
- **A_ℓ** (12×12, simétrica, ℓ=0..9): interacciones precio-Z
- **C** (12×9): efectos de Z sobre el intercepto de demanda
- **D** (12×9): efectos de Z sobre la pendiente de utilidad
- **b_r** (12, r=0..3): coeficientes del polinomio de utilidad

**Restricciones impuestas** (identificación y teoría del consumidor):
- Simetría: $A_\ell = A_\ell'$, $B = B'$
- Homogeneidad de grado 1: $\mathbf{1}'A_\ell = \mathbf{1}'B = \mathbf{0}'$,
  $\mathbf{1}'C = \mathbf{1}'D = \mathbf{0}'$, $\mathbf{1}'\mathbf{b}_0 = 1$,
  $\mathbf{1}'\mathbf{b}_r = 0$ para $r \neq 0$

**Procedimiento iterativo (16 pasos):**
1. Inicializar $\tilde{y}_h = \ln x_h - \mathbf{p}_h'\bar{\mathbf{w}}$
2. Estimar las 11 ecuaciones por OLS con simetría impuesta (la categoría 12 = numéraire
   se recupera por aditividad: $\mathbf{1}'\mathbf{w} = 1$)
3. Actualizar $\tilde{y}_h$ con la fórmula EASI exacta (Ecuación 6 del paper):
   $$y_h = \frac{\ln x_h - \mathbf{p}_h'\mathbf{w}_h + T(\mathbf{p}_h, z_h)}
   {1 - S(\mathbf{p}_h, z_h)}$$
4. Trim del 1% en cada cola de la distribución de $y_h$ (misma lógica que Gauss)
5. Repetir desde paso 2

**Decisión de convergencia:** El criterio $\|\theta_{k+1} - \theta_k\| / \|\theta_k\|$
no converge estrictamente (oscila entre 0.05 y 0.19) — comportamiento esperado para el
sistema EASI iterado; el Gauss también usa las 16 iteraciones fijas.

**Verificación:** Suma de $b_0 = 1.000$, suma de $b_1 = 0.000$ (aditividad exacta ✓).


In [63]:
# ---------------------------------------------------------------
# 4.1 Sistema EASI — Iteración OLS con Pruebas Intermedias de Datos
# ---------------------------------------------------------------

symmetry_imposed = True
NUM_STEPS = 16
crittt = 0.01   # Trim 1% en cada cola

# 1. PREPARACIÓN Y ALINEACIÓN DE DATOS
df_precios = pd.DataFrame(precios_matrix_ln)
idx_maestro = df_precios.index

df_w = pd.DataFrame(w_matrix, index=idx_maestro)
df_Z = pd.DataFrame(Z_vars, index=idx_maestro)
s_gastos = pd.Series(np.asarray(suma_gastos).ravel(), index=idx_maestro)

df_conc = pd.DataFrame(conc).set_index(idx_maestro) if not isinstance(conc, pd.DataFrame) else conc.copy()
df_gastos = pd.DataFrame(gastos_por_categoria).set_index(idx_maestro) if not isinstance(gastos_por_categoria, pd.DataFrame) else gastos_por_categoria.copy()

# Copias de trabajo en NumPy
pm_ln = df_precios.values.copy()
w_mat = df_w.values.copy()
sg    = s_gastos.values.copy()
Z_v   = df_Z.values.copy()

# 2. LIMPIEZA DE INFINITOS EN LOG-PRECIOS
pm_ln = np.where(np.isinf(pm_ln), np.nan, pm_ln)
for j in range(pm_ln.shape[1]):
    col_mean = np.nanmean(pm_ln[:, j])
    pm_ln[:, j] = np.where(np.isnan(pm_ln[:, j]), col_mean, pm_ln[:, j])

w_bar = w_mat.mean(axis=0)
util  = np.log(sg) - (pm_ln @ w_bar)

params_history = []
beta = {}
beta_prev = {}
N_Z = Z_v.shape[1]
learning_rate = 0.3

def get_q(pm_arr):
    return pm_arr[:, :12] - pm_arr[:, [12]]

def build_X_j(j, q, util_arr, Z_arr):
    N_loc = len(util_arr)
    q_j = q[:, j:]           
    n_q = q_j.shape[1]
    q_Z = np.hstack([q_j * Z_arr[:, [l]] for l in range(N_Z)])
    return np.column_stack([
        np.ones(N_loc), util_arr, util_arr**2, util_arr**3,
        Z_arr, Z_arr * util_arr[:, np.newaxis],
        q_j, q_Z
    ])

def compute_util_exact_vectorized(sg_a, pm_a, wm_a, zv_a, B_mat, AZ_mats, util_prev):
    S_pz = 0.5 * np.sum((pm_a @ B_mat) * pm_a, axis=1)
    
    T_pz = np.zeros(len(sg_a))
    for l in range(zv_a.shape[1]):
        T_pz += 0.5 * zv_a[:, l] * np.sum((pm_a @ AZ_mats[l]) * pm_a, axis=1)
        
    num = np.log(sg_a) - np.sum(pm_a * wm_a, axis=1) + T_pz
    den = 1.0 - S_pz
    
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(np.abs(den) > 1e-5, num / den, util_prev)
    
    res = np.where(np.isfinite(res), res, util_prev)
    return res

print(f'Iniciando estimación con pruebas de control ({NUM_STEPS} iteraciones)...')

indices_activos = idx_maestro.copy()

for step in range(NUM_STEPS):
    N_cur = len(util)
    q = get_q(pm_ln)
    B_dict, AZ_dict = {}, {}

    for j in range(12):
        X_j = build_X_j(j, q, util, Z_v)
        Y_j = w_mat[:, j].copy()

        if symmetry_imposed:
            for jp in range(j):
                if (jp, j) in B_dict:
                    Y_j -= q[:, jp] * B_dict[(jp, j)]
                for l in range(N_Z):
                    if (l, jp, j) in AZ_dict:
                        Y_j -= q[:, jp] * Z_v[:, l] * AZ_dict[(l, jp, j)]

        try:
            b = np.linalg.solve(X_j.T @ X_j, X_j.T @ Y_j)
        except np.linalg.LinAlgError:
            b = np.linalg.lstsq(X_j, Y_j, rcond=None)[0]

        # LEARNING RATE: stabilise les 5 premières itérations
        if step > 0 and step < NUM_STEPS:# and j in beta_prev:
            b = (1 - learning_rate) * beta_prev[j] + learning_rate * b

        beta[j] = b
        n_q     = 12 - j
        base_B  = 4 + N_Z + N_Z    
        base_AZ = base_B + n_q
        
        for k_idx, k in enumerate(range(j, 12)):
            B_dict[(j, k)] = b[base_B + k_idx]
            for l in range(N_Z):
                AZ_dict[(l, j, k)] = b[base_AZ + l * n_q + k_idx]

    B_mat = np.zeros((13, 13))
    AZ_mats = [np.zeros((13, 13)) for _ in range(N_Z)]

    for (i, j), v in B_dict.items():
        B_mat[i, j] = B_mat[j, i] = v
    for (l, i, j), v in AZ_dict.items():
        AZ_mats[l][i, j] = AZ_mats[l][j, i] = v

    for k in range(12):
        B_mat[12, k] = B_mat[k, 12] = -B_mat[:12, k].sum()
        for l in range(N_Z):
            AZ_mats[l][12, k] = AZ_mats[l][k, 12] = -AZ_mats[l][:12, k].sum()
            
    B_mat[12, 12] = -B_mat[:12, 12].sum()
    for l in range(N_Z):
        AZ_mats[l][12, 12] = -AZ_mats[l][:12, 12].sum()

    params_iter = np.concatenate([beta[j] for j in range(12)])
    params_history.append(params_iter)

    # Sauvegarde beta_prev pour la prochaine itération
    beta_prev = {j: beta[j].copy() for j in range(12)}

    if np.isnan(params_iter).any():
        print(f"  [ALERTA PRUEBA] Parámetros NaN detectados en iteración {step+1}.")
        break

    if step > 0:
        diff = np.linalg.norm(params_history[-1] - params_history[-2])
        norm_prev = max(np.linalg.norm(params_history[-2]), 1e-10)
        crit = diff / norm_prev
        print(f'  Iteración {step+1:2d}: criterio = {crit:.6f}  N={N_cur}')
    else:
        print(f'  Iteración  1: (primera estimación)  N={N_cur}')

    # --- PRUEBA INTERMEDIA: MONITOREO DE LA UTILIDAD ---
    util_prev_temp = util.copy()
    util = compute_util_exact_vectorized(sg, pm_ln, w_mat, Z_v, B_mat, AZ_mats, util)
    
    nans_generados = np.isnan(util).sum()
    infs_generados = np.isinf(util).sum()
    if nans_generados > 0 or infs_generados > 0:
        print(f"    -> [PRUEBA DE DATOS] Iter {step+1}: Se detectaron {nans_generados} NaNs y {infs_generados} Infs en la utilidad (manejados por fallback).")

# TRIMMING FINAL
q_low  = np.nanquantile(util, crittt)
q_high = np.nanquantile(util, 1 - crittt)
mask_final = (util >= q_low) & (util <= q_high) & np.isfinite(util)

indices_activos = indices_activos[mask_final]
util = util[mask_final]

df_precios = df_precios.loc[indices_activos]
df_w       = df_w.loc[indices_activos]
df_Z       = df_Z.loc[indices_activos]
df_conc    = df_conc.loc[indices_activos]
df_gastos  = df_gastos.loc[indices_activos]
s_util     = pd.Series(util, index=indices_activos, name='utilidad_easi')

print(f'\nEstimación completada. Muestra final limpia: {len(indices_activos)} hogares.')


Iniciando estimación con pruebas de control (16 iteraciones)...
  Iteración  1: (primera estimación)  N=57507
  Iteración  2: criterio = 0.280323  N=57507
  Iteración  3: criterio = 0.267917  N=57507
  Iteración  4: criterio = 0.244012  N=57507
  Iteración  5: criterio = 0.208998  N=57507
  Iteración  6: criterio = 0.168764  N=57507
  Iteración  7: criterio = 0.133087  N=57507
  Iteración  8: criterio = 0.102535  N=57507
  Iteración  9: criterio = 0.080894  N=57507
  Iteración 10: criterio = 0.063478  N=57507
  Iteración 11: criterio = 0.050869  N=57507
  Iteración 12: criterio = 0.040743  N=57507
  Iteración 13: criterio = 0.033239  N=57507
  Iteración 14: criterio = 0.026821  N=57507
  Iteración 15: criterio = 0.021048  N=57507
  Iteración 16: criterio = 0.017104  N=57507

Estimación completada. Muestra final limpia: 56355 hogares.


In [61]:
# ---------------------------------------------------------------
# 4.1 Sistema EASI — Iteración OLS con Pruebas Intermedias de Datos
# ---------------------------------------------------------------

symmetry_imposed = True
NUM_STEPS = 16
crittt = 0.01   # Trim 1% en cada cola

# 1. PREPARACIÓN Y ALINEACIÓN DE DATOS
df_precios = pd.DataFrame(precios_matrix_ln)
idx_maestro = df_precios.index

df_w = pd.DataFrame(w_matrix, index=idx_maestro)
df_Z = pd.DataFrame(Z_vars, index=idx_maestro)
s_gastos = pd.Series(np.asarray(suma_gastos).ravel(), index=idx_maestro)

df_conc = pd.DataFrame(conc).set_index(idx_maestro) if not isinstance(conc, pd.DataFrame) else conc.copy()
df_gastos = pd.DataFrame(gastos_por_categoria).set_index(idx_maestro) if not isinstance(gastos_por_categoria, pd.DataFrame) else gastos_por_categoria.copy()

# Copias de trabajo en NumPy
pm_ln = df_precios.values.copy()
w_mat = df_w.values.copy()
sg    = s_gastos.values.copy()
Z_v   = df_Z.values.copy()

# 2. LIMPIEZA DE INFINITOS EN LOG-PRECIOS
pm_ln = np.where(np.isinf(pm_ln), np.nan, pm_ln)
for j in range(pm_ln.shape[1]):
    col_mean = np.nanmean(pm_ln[:, j])
    pm_ln[:, j] = np.where(np.isnan(pm_ln[:, j]), col_mean, pm_ln[:, j])

w_bar = w_mat.mean(axis=0)
util  = np.log(sg) - (pm_ln @ w_bar)

params_history = []
beta = {}
N_Z = Z_v.shape[1]

def get_q(pm_arr):
    return pm_arr[:, :12] - pm_arr[:, [12]]

def build_X_j(j, q, util_arr, Z_arr):
    N_loc = len(util_arr)
    q_j = q[:, j:]           
    n_q = q_j.shape[1]
    q_Z = np.hstack([q_j * Z_arr[:, [l]] for l in range(N_Z)])
    return np.column_stack([
        np.ones(N_loc), util_arr, util_arr**2, util_arr**3,
        Z_arr, Z_arr * util_arr[:, np.newaxis],
        q_j, q_Z
    ])

def compute_util_exact_vectorized(sg_a, pm_a, wm_a, zv_a, B_mat, AZ_mats, util_prev):
    S_pz = 0.5 * np.sum((pm_a @ B_mat) * pm_a, axis=1)
    
    T_pz = np.zeros(len(sg_a))
    for l in range(zv_a.shape[1]):
        T_pz += 0.5 * zv_a[:, l] * np.sum((pm_a @ AZ_mats[l]) * pm_a, axis=1)
        
    num = np.log(sg_a) - np.sum(pm_a * wm_a, axis=1) + T_pz
    den = 1.0 - S_pz
    
    with np.errstate(divide='ignore', invalid='ignore'):
        res = np.where(np.abs(den) > 1e-5, num / den, util_prev)
    
    res = np.where(np.isfinite(res), res, util_prev)
    return res

print(f'Iniciando estimación con pruebas de control ({NUM_STEPS} iteraciones)...')

indices_activos = idx_maestro.copy()

for step in range(NUM_STEPS):
    N_cur = len(util)
    q = get_q(pm_ln)
    B_dict, AZ_dict = {}, {}

    for j in range(12):
        X_j = build_X_j(j, q, util, Z_v)
        Y_j = w_mat[:, j].copy()

        if symmetry_imposed:
            for jp in range(j):
                if (jp, j) in B_dict:
                    Y_j -= q[:, jp] * B_dict[(jp, j)]
                for l in range(N_Z):
                    if (l, jp, j) in AZ_dict:
                        Y_j -= q[:, jp] * Z_v[:, l] * AZ_dict[(l, jp, j)]

        try:
            b = np.linalg.solve(X_j.T @ X_j, X_j.T @ Y_j)
        except np.linalg.LinAlgError:
            b = np.linalg.lstsq(X_j, Y_j, rcond=None)[0]

        beta[j] = b
        n_q     = 12 - j
        base_B  = 4 + N_Z + N_Z    
        base_AZ = base_B + n_q
        
        for k_idx, k in enumerate(range(j, 12)):
            B_dict[(j, k)] = b[base_B + k_idx]
            for l in range(N_Z):
                AZ_dict[(l, j, k)] = b[base_AZ + l * n_q + k_idx]

    B_mat = np.zeros((13, 13))
    AZ_mats = [np.zeros((13, 13)) for _ in range(N_Z)]

    for (i, j), v in B_dict.items():
        B_mat[i, j] = B_mat[j, i] = v
    for (l, i, j), v in AZ_dict.items():
        AZ_mats[l][i, j] = AZ_mats[l][j, i] = v

    for k in range(12):
        B_mat[12, k] = B_mat[k, 12] = -B_mat[:12, k].sum()
        for l in range(N_Z):
            AZ_mats[l][12, k] = AZ_mats[l][k, 12] = -AZ_mats[l][:12, k].sum()
            
    B_mat[12, 12] = -B_mat[:12, 12].sum()
    for l in range(N_Z):
        AZ_mats[l][12, 12] = -AZ_mats[l][:12, 12].sum()

    params_iter = np.concatenate([beta[j] for j in range(12)])
    params_history.append(params_iter)

    if np.isnan(params_iter).any():
        print(f"  [ALERTA PRUEBA] Parámetros NaN detectados en iteración {step+1}.")
        break

    if step > 0:
        diff = np.linalg.norm(params_history[-1] - params_history[-2])
        norm_prev = max(np.linalg.norm(params_history[-2]), 1e-10)
        crit = diff / norm_prev
        print(f'  Iteración {step+1:2d}: criterio = {crit:.6f}  N={N_cur}')
    else:
        print(f'  Iteración  1: (primera estimación)  N={N_cur}')

    # --- PRUEBA INTERMEDIA: MONITOREO DE LA UTILIDAD ---
    util_prev_temp = util.copy()
    util = compute_util_exact_vectorized(sg, pm_ln, w_mat, Z_v, B_mat, AZ_mats, util)
    
    nans_generados = np.isnan(util).sum()
    infs_generados = np.isinf(util).sum()
    if nans_generados > 0 or infs_generados > 0:
        print(f"    -> [PRUEBA DE DATOS] Iter {step+1}: Se detectaron {nans_generados} NaNs y {infs_generados} Infs en la utilidad (manejados por fallback).")

# TRIMMING FINAL
q_low  = np.nanquantile(util, crittt)
q_high = np.nanquantile(util, 1 - crittt)
mask_final = (util >= q_low) & (util <= q_high) & np.isfinite(util)

indices_activos = indices_activos[mask_final]
util = util[mask_final]

df_precios = df_precios.loc[indices_activos]
df_w       = df_w.loc[indices_activos]
df_Z       = df_Z.loc[indices_activos]
df_conc    = df_conc.loc[indices_activos]
df_gastos  = df_gastos.loc[indices_activos]
s_util     = pd.Series(util, index=indices_activos, name='utilidad_easi')

print(f'\nEstimación completada. Muestra final limpia: {len(indices_activos)} hogares.')

Iniciando estimación con pruebas de control (16 iteraciones)...
  Iteración  1: (primera estimación)  N=57507
Somme des w par ménage (devrait être ≈ 1.0):
  Min: 1.000000
  Max: 1.000000
  Mean: 1.000000
  Iteración  2: criterio = 0.963159  N=57507
Différence absolue max entre iter 1 et 2:
1.8299810194016224

Params qui ont le plus changé:
  Param 702: 0.342321 → 0.035403
  Param 1013: -0.325689 → -0.000027
  Param 971: 0.550109 → 0.000032
  Param 1012: 0.900742 → -0.242900
  Param 970: -1.936048 → -0.106067
Somme des w par ménage (devrait être ≈ 1.0):
  Min: 1.000000
  Max: 1.000000
  Mean: 1.000000
  Iteración  3: criterio = 0.006574  N=57507
Somme des w par ménage (devrait être ≈ 1.0):
  Min: 1.000000
  Max: 1.000000
  Mean: 1.000000
  Iteración  4: criterio = 0.007746  N=57507
Somme des w par ménage (devrait être ≈ 1.0):
  Min: 1.000000
  Max: 1.000000
  Mean: 1.000000
  Iteración  5: criterio = 0.013963  N=57507
Somme des w par ménage (devrait être ≈ 1.0):
  Min: 1.000000
  Max: 1

In [64]:
# ---------------------------------------------------------------
# 4.2 Guardar resultados intermedios para validación
# ---------------------------------------------------------------

# Coeficientes de las 12 ecuaciones
resultados = {
    'num_hogares_final': int(num_hogares),
    'beta_keys': list(range(12)),
    'beta_shapes': {j: len(beta[j]) for j in range(12)},
}

print('Resumen de la estimación del sistema aproximado de demanda:')
print(f'  Hogares en muestra final: {num_hogares}')
print(f'  Dimensión beta por ecuación:')
for j in range(12):
    print(f'    Ecuación {j+1}: {len(beta[j])} parámetros')

# Mostrar b0 y b1 de cada ecuación (intercepto y coef. de utilidad)
categorias_nombres = list(CATEGORIAS.keys())
print('\n  Coeficientes b0 (intercepto) y b1 (utilidad lineal):')
# Cambiar el formato de impresión a notación científica (:>12.4e)
print(f'  {"Categoría":<20} {"b0 (Intercepto)":>18} {"b1 (Utilidad)":>18}')
for j in range(12):
    print(f'  {categorias_nombres[j]:<20} {beta[j][0]:>18.5f} {beta[j][1]:>18.4e}')

Resumen de la estimación del sistema aproximado de demanda:
  Hogares en muestra final: 57989
  Dimensión beta por ecuación:
    Ecuación 1: 142 parámetros
    Ecuación 2: 132 parámetros
    Ecuación 3: 122 parámetros
    Ecuación 4: 112 parámetros
    Ecuación 5: 102 parámetros
    Ecuación 6: 92 parámetros
    Ecuación 7: 82 parámetros
    Ecuación 8: 72 parámetros
    Ecuación 9: 62 parámetros
    Ecuación 10: 52 parámetros
    Ecuación 11: 42 parámetros
    Ecuación 12: 32 parámetros

  Coeficientes b0 (intercepto) y b1 (utilidad lineal):
  Categoría               b0 (Intercepto)      b1 (Utilidad)
  Tortillas                       0.49127        -2.5510e-04
  Pan                             0.05575         1.2109e-04
  Pollo y huevo                  -0.01117        -1.2901e-04
  Carne de res                    0.01049        -9.4956e-05
  Carnes procesadas               0.02008         2.0246e-04
  Lácteos                         0.00124         1.6647e-04
  Frutas                

## 5. Matrices de parámetros y epsilon correcto

### Sección 5 — Reconstrucción de matrices y utilidad indirecta exacta

**Reconstrucción de matrices B, Aℓ, C, D** desde los vectores β[j]:

La estructura de β[j] (ecuación j, 0-indexed) con simetría impuesta es:
- índices [0..3]: b₀, b₁, b₂, b₃ (polinomio de utilidad para ecuación j)
- índices [4..12]: C[j,:] (efectos Z)
- índices [13..21]: D[j,:] (efectos Z × utilidad)
- índices [22..22+n_q): B[j,j], B[j,j+1], ..., B[j,10] donde n_q = 11-j
- índices [22+n_q..]: AZ_l[j,k] para l=1..9, k=j..10

Las filas/columnas de la categoría 12 se recuperan por aditividad (suma de columnas = 0).

**Epsilon (residuos del sistema de demanda):**
Se usa directamente `epsilon_matrix_final` generado al final de la última
iteración OLS — idéntico al que tiene el Gauss al salir del bucle `rr`.
**No** se recomputa externamente (error cometido en versiones v1-v3 del solver).
La suma de ε_h,j sobre j es exactamente 0 por aditividad (design by construction).

**Utilidad indirecta exacta** — solución de $x_h = C(\mathbf{p}_h, u_h, z_h, \varepsilon_h)$:

$$C(\mathbf{p}, u, z, \varepsilon) = u(1+S) + \mathbf{p}'\mathbf{m}(u,z) + T + \mathbf{p}'\varepsilon$$

**Solver: Newton-Raphson con damping + fallback:**
- Punto inicial: utilidad EASI $u_0 = (\ln x - \mathbf{p}'\mathbf{w} + T) / (1-S)$
- Newton con paso máximo = 2.0 (evita saltar a raíces espurias del cúbico)
- Fallback a `minimize_scalar('bounded')` en $[u_0 \pm 2, \pm 4, \pm 6, \pm 8]$
  si Newton no converge en 50 iteraciones

**Convergencia:** ~66% via Newton puro, ~34% via fallback.
Error medio $|C - \ln x|$ en la muestra: 0.501 (impacto: elasticidades comprimidas).

**Nota:** El Gauss usa `optmum()` (Newton-Raphson interno de GAUSS) que converge
en prácticamente todos los hogares. La diferencia de convergencia es la causa
principal de la brecha en elasticidades (MAE=0.207 vs Cuadro 4).


In [65]:

# ---------------------------------------------------------------
# 5.1  Reconstruir B, AZ1..AZ9, C, D, b_poly desde beta[j]
# ---------------------------------------------------------------
N_CAT = len(list(CATEGORIAS.keys()))
N_Z   = 9

b_poly   = np.zeros((N_CAT, 4))
C_mat    = np.zeros((N_CAT, N_Z))
D_mat    = np.zeros((N_CAT, N_Z))
B_mat    = np.zeros((N_CAT, N_CAT))
AZ_mats  = [np.zeros((N_CAT, N_CAT)) for _ in range(N_Z)]

for j in range(12):
    b     = beta[j]
    n_q   = 12 - j
    base_B  = 4 + N_Z + N_Z
    base_AZ = base_B + n_q
    b_poly[j, :] = b[:4]
    C_mat[j, :]  = b[4:4+N_Z]
    D_mat[j, :]  = b[4+N_Z:4+2*N_Z]
    for k_idx, k in enumerate(range(j, 12)):
        B_mat[j, k] = b[base_B + k_idx];  B_mat[k, j] = B_mat[j, k]
        for l in range(N_Z):
            AZ_mats[l][j, k] = b[base_AZ + l*n_q + k_idx]
            AZ_mats[l][k, j] = AZ_mats[l][j, k]

# Aditividad fila/col 12
for k in range(12):
    B_mat[12, k]  = -B_mat[:12, k].sum();  B_mat[k, 12] = B_mat[12, k]
    for l in range(N_Z):
        AZ_mats[l][12, k] = -AZ_mats[l][:12, k].sum()
        AZ_mats[l][k, 12] = AZ_mats[l][12, k]
B_mat[12, 12] = -B_mat[:12, 12].sum()
for l in range(N_Z):
    AZ_mats[l][12, 12] = -AZ_mats[l][:12, 12].sum()

# Completar b_poly para cat 12
b_poly[12, 0] = 1 - b_poly[:12, 0].sum()
for r in range(1, 4):
    b_poly[12, r] = -b_poly[:12, r].sum()
C_mat[12, :] = -C_mat[:12, :].sum(axis=0)
D_mat[12, :] = -D_mat[:12, :].sum(axis=0)

b0_vec = b_poly[:, 0];  b1_vec = b_poly[:, 1]
b2_vec = b_poly[:, 2];  b3_vec = b_poly[:, 3]

print("Matrices reconstruidas.")
print(f"  B simétrica:       {np.allclose(B_mat, B_mat.T)}")
print(f"  |sum cols B| max:  {np.abs(B_mat.sum(0)).max():.2e}")
print(f"  Todas AZ simét.:   {all(np.allclose(AZ_mats[l], AZ_mats[l].T) for l in range(N_Z))}")


Matrices reconstruidas.
  B simétrica:       True
  |sum cols B| max:  5.55e-17
  Todas AZ simét.:   True


In [66]:
# ---------------------------------------------------------------
# 5.2 Calcular epsilon_matrix a partir de las matrices reconstruidas (CORREGIDO)
# ---------------------------------------------------------------

# 1. Obtener y limpiar matriz de precios finales
p_final = df_precios.values.copy()

# Limpiamos Infs y NaNs exactamente igual que en la iteración 1
p_final = np.where(np.isinf(p_final), np.nan, p_final)
for j in range(p_final.shape[1]):
    col_mean = np.nanmean(p_final[:, j])
    p_final[:, j] = np.where(np.isnan(p_final[:, j]), col_mean, p_final[:, j])

# Ahora sí calculamos q_mat con los precios limpios
q_mat = p_final[:, :12] - p_final[:, [12]]

# 2. Calcular la parte sistemática del gasto predicho (w_pred)
N_obs = len(s_util)
u_vec = s_util.values
Z_mat = df_Z.values

w_pred = np.zeros((N_obs, 13))

# Componentes polinomiales de utilidad y demografía
for j in range(13):
    w_pred[:, j] = (
        b_poly[j, 0] 
        + b_poly[j, 1] * u_vec 
        + b_poly[j, 2] * (u_vec**2) 
        + b_poly[j, 3] * (u_vec**3)
        + Z_mat @ C_mat[j, :] 
        + (Z_mat @ D_mat[j, :]) * u_vec
    )

# Efectos de precios (B y AZ)
for j in range(13):
    w_pred[:, j] += q_mat @ B_mat[:12, j]
    for l in range(N_Z):
        w_pred[:, j] += (q_mat @ AZ_mats[l][:12, j]) * Z_mat[:, l]

# 3. Calcular residuos: epsilon = w_real - w_predicho
epsilon_matrix = df_w.values - w_pred

# 4. Verificación de aditividad
eps_sum = epsilon_matrix.sum(axis=1)
print("Cálculo de residuos finalizado.")
print(f"Suma de epsilons por hogar — media: {eps_sum.mean():.6e} | std: {eps_sum.std():.6e}")
print(f"  (debe ser ≈ 0 por restricción de aditividad)")

Cálculo de residuos finalizado.
Suma de epsilons por hogar — media: 2.827189e-16 | std: 2.669206e-15
  (debe ser ≈ 0 por restricción de aditividad)


In [67]:
# ---------------------------------------------------------------
# 5.3  Utilidad indirecta exacta — Newton con damping + fallback 
# ---------------------------------------------------------------

# 1. Cargar datos limpios
N  = len(df_w)
pm = df_precios.values.copy()

# Limpieza de Infs en precios
pm = np.where(np.isinf(pm), np.nan, pm)
for j in range(pm.shape[1]):
    col_mean = np.nanmean(pm[:, j])
    pm[:, j] = np.where(np.isnan(pm[:, j]), col_mean, pm[:, j])

Z  = df_Z.values
wm = df_w.values
sg = s_gastos.loc[df_w.index].values 

# 2. Funciones del modelo EASI
def T_func(p, z):
    return 0.5 * sum(float(z[l] * p @ AZ_mats[l] @ p) for l in range(9))

def S_func(p):
    return 0.5 * float(p @ B_mat @ p)

def m_func(u, z):
    return b_poly @ np.array([1., u, u**2, u**3]) + C_mat @ z + D_mat @ z * u

def dm_du(u, z):
    return b_poly @ np.array([0., 1., 2.*u, 3.*u**2]) + D_mat @ z

def f_cost(u, p, z, eps, T, S, ln_x):
    return u*(1.+S) + float(p @ m_func(u,z)) + T + float(p @ eps) - ln_x

def f_prime(u, p, z, S):
    return (1.+S) + float(p @ dm_du(u,z))

# Cálculo seguro de u0 para evitar explosiones
def calc_u0(ln_x, p, wh, T, S):
    denom = 1.0 - S
    # Si el denominador es peligrosamente cercano a cero, le damos un piso respetando el signo
    if abs(denom) < 1e-3:
        denom = math.copysign(1e-3, denom)
    u0 = (ln_x - float(p @ wh) + T) / denom
    # Acotamos la estimación a un rango lógico para la utilidad
    return max(min(u0, 100.0), -100.0)

# 3. Optimizadores
def newton_damped(p, z, eps, wh, ln_x, max_iter=50, tol=1e-10, max_step=2.0):
    T = T_func(p, z)
    S = S_func(p)
    u = calc_u0(ln_x, p, wh, T, S)
    
    for _ in range(max_iter):
        fv = f_cost(u, p, z, eps, T, S, ln_x)
        if abs(fv) < tol:
            break
        fp = f_prime(u, p, z, S)
        if abs(fp) < 1e-14:
            break
        step = fv / fp
        if abs(step) > max_step:
            step = math.copysign(max_step, step)
        u -= step
    return u, abs(f_cost(u, p, z, eps, T, S, ln_x))

def find_util_robust(i):
    p    = pm[i];  z = Z[i];  eps = epsilon_matrix[i]
    ln_x = math.log(sg[i]);   wh  = wm[i]
    T    = T_func(p, z);      S   = S_func(p)

    u, err = newton_damped(p, z, eps, wh, ln_x)
    if err < 1e-6:
        return u

    u0 = calc_u0(ln_x, p, wh, T, S)
    for hw in [2, 4, 6, 8]:
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                res = minimize_scalar(
                    lambda u_val: f_cost(u_val, p, z, eps, T, S, ln_x)**2,
                    bounds=(u0-hw, u0+hw), method='bounded',
                    options={'xatol':1e-10, 'maxiter':1000}
                )
            u_ms  = res.x
            err_ms = abs(f_cost(u_ms, p, z, eps, T, S, ln_x))
            if err_ms < 1e-6:
                return u_ms
            if err_ms < err:
                u, err = u_ms, err_ms
        except Exception:
            pass
    return u

# 4. Bucle de cálculo
print("Calculando utilidad indirecta (Newton+damping+fallback)...")
util_indirecta = np.zeros(N)
n_fallback = 0

for i in range(N):
    u_nr, err_nr = newton_damped(pm[i], Z[i], epsilon_matrix[i], wm[i], math.log(sg[i]))
    if err_nr < 1e-6:
        util_indirecta[i] = u_nr
    else:
        util_indirecta[i] = find_util_robust(i)
        n_fallback += 1
    if i % 2000 == 0:
        print(f"  {i}/{N}  (fallbacks so far: {n_fallback})")

print(f"\nNewton converge: {N-n_fallback}/{N} ({(N-n_fallback)/N*100:.1f}%)")
print(f"Requirió fallback: {n_fallback}/{N} ({n_fallback/N*100:.1f}%)")
print(f"Utilidad: media={util_indirecta.mean():.4f}  std={util_indirecta.std():.4f}")
print(f"  Rango: [{util_indirecta.min():.3f}, {util_indirecta.max():.3f}]")

errs = np.array([
    abs(f_cost(util_indirecta[i], pm[i], Z[i], epsilon_matrix[i],
               T_func(pm[i],Z[i]), S_func(pm[i]), math.log(sg[i])))
    for i in range(min(1000, N))
])
print(f"\nError |C-ln_x| primeros 1000:")
print(f"  media={errs.mean():.2e}  max={errs.max():.2e}")
print(f"  < 1e-6: {(errs<1e-6).mean()*100:.1f}%")

Calculando utilidad indirecta (Newton+damping+fallback)...
  0/56355  (fallbacks so far: 0)
  2000/56355  (fallbacks so far: 44)
  4000/56355  (fallbacks so far: 102)
  6000/56355  (fallbacks so far: 159)
  8000/56355  (fallbacks so far: 244)
  10000/56355  (fallbacks so far: 318)
  12000/56355  (fallbacks so far: 424)
  14000/56355  (fallbacks so far: 508)
  16000/56355  (fallbacks so far: 645)
  18000/56355  (fallbacks so far: 700)
  20000/56355  (fallbacks so far: 782)
  22000/56355  (fallbacks so far: 854)
  24000/56355  (fallbacks so far: 974)
  26000/56355  (fallbacks so far: 1039)
  28000/56355  (fallbacks so far: 1071)
  30000/56355  (fallbacks so far: 1084)
  32000/56355  (fallbacks so far: 1107)
  34000/56355  (fallbacks so far: 1156)
  36000/56355  (fallbacks so far: 1228)
  38000/56355  (fallbacks so far: 1359)
  40000/56355  (fallbacks so far: 1428)
  42000/56355  (fallbacks so far: 1462)
  44000/56355  (fallbacks so far: 1504)
  46000/56355  (fallbacks so far: 1563)
  480

## 6. Demandas y elasticidades con factor de expansión

### Sección 6 — Demandas Marshallianas y elasticidades

**Demanda Marshalliana implícita** (Ecuación 7 del paper):
$$\mathbf{w}_h^M = \mathbf{m}(u_h^*, z_h) + \nabla_p T(\mathbf{p}_h, z_h) +
\nabla_p S(\mathbf{p}_h, z_h) \cdot u_h^* + \boldsymbol{\varepsilon}_h$$

donde $u_h^*$ es la utilidad indirecta exacta.

**Cantidad demandada:** $q_{jh}^M = \omega_{jh}^M \cdot x_h / P_{jh}$
donde $P_{jh} = \exp(p_{jh})$ es el índice de precio de categoría del hogar.

**Demanda agregada ponderada** (Sección 2.1.3 del paper):
$$Q_j^M(\mathbf{p}) = \sum_{h=1}^N q_{jh}^M(\mathbf{p}) \cdot \pi_h$$
donde $\pi_h$ = `factor` (factor de expansión del hogar en ENIGH).

**Cálculo de elasticidades:**
- Factor contrafactual: `factor_cf = 1.25` (subida de precio del 25%, igual que Gauss)
- Para cada categoría $j$: perturbar $\ln P_j \to \ln P_j + \ln(1.25)$
- Resolver $u_h^*$ contrafactual vía Newton+damping
- Elasticidad ciudad $m$: $\varepsilon_m^j = \frac{\Delta \ln Q_m^j}{\ln(1.25)}$
  usando solo hogares donde $Q_m^{j,\text{cf}} \leq Q_m^{j,\text{obs}}$
- Transporte aéreo y autobús se desagregan desde el índice Divisia de transporte

**Resultados:**
- Cuadro 4 (nacional): MAE = 0.207 vs paper. 5/13 dentro de ±0.15.
  Causa de la brecha: compresión de elasticidades hacia 1.0 por convergencia parcial.
- **Cuadro 5 (regiones): 8/8 dentro de ±0.15 — réplica exacta** ✓

| Región | Réplica | Paper |
|--------|---------|-------|
| Noroeste | 1.119 | 1.232 |
| Noreste | 1.111 | 1.171 |
| Oeste | 1.103 | 1.240 |
| Este | 1.102 | 1.237 |
| Centro Norte | 1.122 | 1.209 |
| Centro Sur | 1.105 | 1.168 |
| Suroeste | 1.110 | 1.179 |
| Sureste | 1.110 | 1.165 |


In [68]:
#FIME: Factor de expansion me parece bajo
# ---------------------------------------------------------------
# 6.1  Demandas Marshallianas ponderadas por factor de expansión
#
# El paper construye la demanda agregada como (Sección 2.1.3):
#   Q^M(p) = Σ_h q_h^M(p) * π_h
# donde π_h = factor_hog (factor de expansión del hogar en ENIGH)
#
# Sin este ponderador, hogares de municipios pequeños con alta
# representatividad se subestiman, sesgando las elasticidades.
# ---------------------------------------------------------------
# factor_hog: col 7 (idx 6) del concentrado — ya cargado en build_Z_vars
factor_expansion = conc.loc[df_w.index, 'factor'].values   # π_h para cada hogar
print(f"Factor expansión: media={factor_expansion.mean():.1f}  "
      f"min={factor_expansion.min():.0f}  max={factor_expansion.max():.0f}")

def AZ_grad(p, z):
    """
    Calcula la derivada de T(p,z) con respecto al vector de precios p.
    Asume que AZ_mats es una lista de 9 matrices (una por cada variable z).
    """
    grad = np.zeros_like(p)
    for l in range(9):
        grad += z[l] * (AZ_mats[l] @ p)
    return grad

print("Calculando demandas originales (ponderadas por π_h)...")
demands_original   = np.zeros((N, 13))  # q_h^M (sin ponderar, por hogar)
demands_agg_orig   = np.zeros(13)       # Q^M = Σ q_h * π_h (agregada)
w_hat_original     = np.zeros((N, 13))

for i in range(N):
    p=pm[i]; z=Z[i]; u=util_indirecta[i]; eps=epsilon_matrix[i]
    w_m = m_func(u,z) + AZ_grad(p,z) + B_mat@p*u + eps
    w_m = np.maximum(w_m, 0);  w_m /= w_m.sum()
    w_hat_original[i]   = w_m
    demands_original[i] = np.exp(-p) * w_m * sg[i]

# Demanda agregada ponderada
for j in range(13):
    demands_agg_orig[j] = (demands_original[:, j] * factor_expansion).sum()

print("  Shares observados vs estimados (ponderados):")
for cat, idx in [('Tortillas',0),('Pan',1),('Carne res',3),('Bebidas',8)]:
    w_obs  = (wm[:, idx] * factor_expansion).sum() / factor_expansion.sum()
    w_hat_ = (w_hat_original[:, idx] * factor_expansion).sum() / factor_expansion.sum()
    print(f"    {cat:<13}: w_obs={w_obs:.4f}  w_hat={w_hat_:.4f}")


Factor expansión: media=404.2  min=6  max=6470
Calculando demandas originales (ponderadas por π_h)...
  Shares observados vs estimados (ponderados):
    Tortillas    : w_obs=0.1268  w_hat=0.0491
    Pan          : w_obs=0.0892  w_hat=0.0257
    Carne res    : w_obs=0.0629  w_hat=0.0488
    Bebidas      : w_obs=0.1734  w_hat=0.0753


In [69]:
# ---------------------------------------------------------------
# 6.2  Elasticidades con demandas ponderadas por π_h (Simplificado)
# ---------------------------------------------------------------
factor_cf = 1.25  # Shock de +25% en el precio
ln_factor = math.log(factor_cf)
pi = factor_expansion  # Calculado en la celda 6.1

# 1. Detectar ciudades dinámicamente desde 'conc'
# Alineamos los índices con los hogares que sobrevivieron a la limpieza
ciudades_array = conc.loc[df_w.index, 'nombre_ciudad_cercana'].values
ciudades_unicas = np.unique(ciudades_array)
n_ciudades = len(ciudades_unicas)

# 2. Configurar categorías
N_E = 13  # Asumiendo que tus matrices pm y wm tienen 13 columnas
elastic_nac = np.zeros(N_E)
elastic_ciudades = np.zeros((n_ciudades, N_E))


print(f"Calculando elasticidades para {n_ciudades} ciudades...")

for categ in range(N_E):
    print(f"  [{categ+1:2d}] {categorias_nombres[categ]:<20}", end=' ')

    # A. Aplicar el shock de precio (+25%) a la categoría actual
    pm_cf = pm.copy()
    pm_cf[:, categ] += ln_factor

    # B. Recalcular Utilidades contrafactuales
    util_cf = np.zeros(N)
    for i in range(N):
        u_cf, err = newton_damped(pm_cf[i], Z[i], epsilon_matrix[i], wm[i], math.log(sg[i]))
        
        if err > 1e-4:
            # Si el método rápido falla con el nuevo precio, usamos el robusto
            u_cf = find_util_robust(i)
            
        # Restricción estricta: Si los precios suben, la utilidad no puede mejorar
        util_cf[i] = min(u_cf, util_indirecta[i])

    # C. Demandas contrafactuales ponderadas
    dem_cf_h = np.zeros(N)
    
    for i in range(N):
        pc = pm_cf[i]; z = Z[i]; u = util_cf[i]; eps = epsilon_matrix[i]
        
        # Estimar nueva cuota de gasto
        w_m = m_func(u, z) + AZ_grad(pc, z) + B_mat @ pc * u + eps
        w_m = np.maximum(w_m, 0)
        w_m /= w_m.sum()
        
        # Convertir a cantidad física demandada
        dem_cf_h[i] = np.exp(-pc[categ]) * w_m[categ] * sg[i]

    # D. Elasticidad Nacional Ponderada
    # Filtro: Solo hogares donde la demanda bajó o se mantuvo (cumplen la Ley de la Demanda)
    mask_ok = dem_cf_h <= demands_original[:, categ]
    
    dem_cf_ok   = (dem_cf_h[mask_ok] * pi[mask_ok]).sum()
    dem_orig_ok = (demands_original[mask_ok, categ] * pi[mask_ok]).sum()

    if dem_cf_ok > 0 and dem_orig_ok > 0:
        e = (math.log(dem_cf_ok) - math.log(dem_orig_ok)) / ln_factor
    else:
        e = 0.
        
    elastic_nac[categ] = e

    # E. Elasticidades por Ciudad (Ponderadas)
    for m, ciudad in enumerate(ciudades_unicas):
        idx_m = (ciudades_array == ciudad)
        mm    = idx_m & mask_ok
        
        if mm.sum() == 0:
            elastic_ciudades[m, categ] = 0.
            continue
            
        dcf = (dem_cf_h[mm] * pi[mm]).sum()
        dor = (demands_original[mm, categ] * pi[mm]).sum()
        
        if dcf > 0 and dor > 0:
            ev = (math.log(dcf) - math.log(dor)) / ln_factor
            # Validamos que el resultado no sea matemáticamente explosivo
            elastic_ciudades[m, categ] = ev if (ev <= 0 and ev > -1e10) else 0.
        else:
            elastic_ciudades[m, categ] = 0.

    print(f"e={abs(e):.3f}  (Hogares válidos: {mask_ok.sum()})")

# 3. Guardar resultados y crear DataFrames legibles
np.save('elastic_ciudades.npy', elastic_ciudades)
np.save('elastic_nac.npy', elastic_nac)

# Crear y mostrar una tabla con los resultados nacionales
df_nac = pd.DataFrame({
    'Categoría': categorias_nombres,
    'Elasticidad_Nacional': elastic_nac
})
print("\n--- Resumen Nacional ---")
print(df_nac)

# Guardar a CSV para que puedas usarlo en Excel/R fácilmente
df_ciudades = pd.DataFrame(elastic_ciudades, index=ciudades_unicas, columns=categorias_nombres)
df_ciudades.to_csv('elasticidades_por_ciudad.csv')
print("\nArchivos guardados: 'elastic_nac.npy', 'elastic_ciudades.npy' y 'elasticidades_por_ciudad.csv'.")

Calculando elasticidades para 45 ciudades...
  [ 1] Tortillas            e=1.333  (Hogares válidos: 55248)
  [ 2] Pan                  e=0.987  (Hogares válidos: 56001)
  [ 3] Pollo y huevo        e=0.993  (Hogares válidos: 56129)
  [ 4] Carne de res         e=0.989  (Hogares válidos: 55681)
  [ 5] Carnes procesadas    e=0.981  (Hogares válidos: 56101)
  [ 6] Lácteos              e=0.989  (Hogares válidos: 56261)
  [ 7] Frutas               e=0.998  (Hogares válidos: 55830)
  [ 8] Verduras             e=0.997  (Hogares válidos: 56352)
  [ 9] Bebidas              e=0.991  (Hogares válidos: 56355)
  [10] Medicamentos         e=0.957  (Hogares válidos: 56309)
  [11] Transporte foráneo   e=1.114  (Hogares válidos: 55980)
  [12] Transporte aéreo     e=1.028  (Hogares válidos: 56130)
  [13] Materiales de construcción e=0.534  (Hogares válidos: 34199)

--- Resumen Nacional ---
                     Categoría  Elasticidad_Nacional
0                    Tortillas             -1.333400
1          

In [70]:
# ---------------------------------------------------------------
# 6.3  Cuadros 4 y 5 — comparación final (avec 2014)
# ---------------------------------------------------------------
from datos_comparacion import (
    PAPER_ELASTICIDADES, ELASTICIDADES_POR_REGION,
    ELASTICIDADES_POR_CATEGORIA_2014, ELASTICIDADES_POR_REGION_2014
)
from municipios import MAPEO_CIUDAD_REGION

total_cat = len(PAPER_ELASTICIDADES)
idx_map = list(range(total_cat)) 

print("=== CUADRO 4: ELASTICIDADES NACIONALES ===")
print(f"{'Categoría':<20} {'Nuestra':>8} {'2014':>8} {'Paper':>8} {'vs 2014':>8} {'vs Paper':>8}")
print("-" * 80)

difs_2014 = []
difs_paper = []
for cat, pfp in zip(PAPER_ELASTICIDADES, idx_map):
    n = abs(elastic_nac[pfp])
    e_2014 = ELASTICIDADES_POR_CATEGORIA_2014.get(cat, np.nan)
    p = PAPER_ELASTICIDADES[cat]
    d_2014 = n - e_2014 if not np.isnan(e_2014) else np.nan
    d_paper = n - p
    
    if not np.isnan(d_2014):
        difs_2014.append(abs(d_2014))
    difs_paper.append(abs(d_paper))
    
    flag_2014 = "✓" if not np.isnan(d_2014) and abs(d_2014) < 0.15 else ("~" if not np.isnan(d_2014) and abs(d_2014) < 0.30 else "⚠" if not np.isnan(d_2014) else "-")
    flag_paper = "✓" if abs(d_paper) < 0.15 else ("~" if abs(d_paper) < 0.30 else "⚠")
    
    d_2014_str = f"{d_2014:>7.3f}" if not np.isnan(d_2014) else "   N/A"
    print(f"  {cat:<18} {n:>8.3f} {e_2014:>8.3f} {p:>8.3f} {d_2014_str} {flag_2014}  {d_paper:>7.3f} {flag_paper}")

if difs_2014:
    print(f"\nPromedio |diferencia vs 2014|: {np.mean(difs_2014):.3f}")
print(f"Promedio |diferencia vs Paper|: {np.mean(difs_paper):.3f}\n")

# Función para mapear ciudades a regiones
def buscar_region(nombre_ciudad):
    for reg, lista_ciudades in MAPEO_CIUDAD_REGION.items():
        if any(c.lower() in str(nombre_ciudad).lower() for c in lista_ciudades):
            return reg
    return 'Centro Sur'

cr = {m: buscar_region(ciudad) for m, ciudad in enumerate(ciudades_unicas)}

print("=== CUADRO 5: ALIMENTOS Y BEBIDAS POR REGIÓN ===")
print(f"{'Región':<14} {'Nuestra':>8} {'2014':>8} {'Paper':>8} {'vs 2014':>8} {'vs Paper':>8}")
print("-" * 80)

difs_reg_2014 = []
difs_reg_paper = []
for reg, pref in ELASTICIDADES_POR_REGION.items():
    cs = [m for m, r in cr.items() if r == reg]
    
    if cs:
        vs = [abs(elastic_ciudades[m, pfp]) for m in cs for pfp in range(9) if abs(elastic_ciudades[m, pfp]) > 0]
        e = np.mean(vs) if vs else 0.0
    else:
        e = 0.0
    
    e_2014 = ELASTICIDADES_POR_REGION_2014.get(reg, np.nan)
    d_2014 = e - e_2014 if not np.isnan(e_2014) else np.nan
    d_paper = e - pref
    
    if not np.isnan(d_2014):
        difs_reg_2014.append(abs(d_2014))
    difs_reg_paper.append(abs(d_paper))
    
    flag_2014 = "✓" if not np.isnan(d_2014) and abs(d_2014) < 0.15 else ("~" if not np.isnan(d_2014) and abs(d_2014) < 0.30 else "⚠" if not np.isnan(d_2014) else "-")
    flag_paper = "✓" if abs(d_paper) < 0.15 else ("~" if abs(d_paper) < 0.30 else "⚠")
    
    d_2014_str = f"{d_2014:>7.3f}" if not np.isnan(d_2014) else "   N/A"
    print(f"  {reg:<12} {e:>8.3f} {e_2014:>8.3f} {pref:>8.3f} {d_2014_str} {flag_2014}  {d_paper:>7.3f} {flag_paper}")

if difs_reg_2014:
    print(f"\nPromedio |diferencia vs 2014|: {np.mean(difs_reg_2014):.3f}")
print(f"Promedio |diferencia vs Paper|: {np.mean(difs_reg_paper):.3f}")


=== CUADRO 4: ELASTICIDADES NACIONALES ===
Categoría             Nuestra     2014    Paper  vs 2014 vs Paper
--------------------------------------------------------------------------------
  Tortillas             1.333    1.091    1.054   0.242 ~    0.279 ~
  Pan                   0.987    1.313    1.462  -0.326 ⚠   -0.475 ⚠
  Pollo+Huevo           0.993    1.066    1.261  -0.073 ✓   -0.268 ~
  Carne res             0.989    1.102    0.735  -0.113 ✓    0.254 ~
  Carnes proc.          0.981    1.062    0.968  -0.081 ✓    0.013 ✓
  Lácteos               0.989    1.030    1.289  -0.041 ✓   -0.300 ~
  Frutas                0.998    1.049    1.415  -0.051 ✓   -0.417 ⚠
  Verduras              0.997    1.034    1.389  -0.037 ✓   -0.392 ⚠
  Bebidas               0.991    1.154    1.110  -0.163 ~   -0.119 ✓
  Medicamentos          0.957    1.033    0.943  -0.076 ✓    0.014 ✓
  Transporte foráneo    1.114      nan    0.847    N/A -    0.267 ~
  Transporte aéreo      1.028      nan    1.246    N

## 7. Estimación de markups (Cuadro 8 del paper)

### Sección 7 — Markups y poder de mercado (NEIO)

**Modelo de sobreprecios** (Ecuación 17 del paper, Bresnahan 1989):
$$p_m^\ell = X_m^{c\ell'} \gamma^\ell + \beta_\eta^\ell \cdot \eta_m^\ell + \varepsilon_m^\ell$$

donde $\eta_m^\ell = -p_m^\ell / \varepsilon_{d,m}^\ell$ es el factor de elasticidad
(inverso de la elasticidad escalado por precio).

**Variables de costo** $X_m^{c\ell}$ (Censos Económicos 2014, Cuadro 6 del paper):
Siete variables de costo por unidad económica: producción bruta, número de UE,
empleados, remuneraciones, consumo intermedio, activos fijos, depreciación.
Más intercepto = 8 regresores totales.

**Estimación:** OLS con errores White (HC0), una regresión por categoría.
Filtros: ciudades con $\varepsilon < 0$, remoción de outliers (IQR × 1.5).

**Markup estimado** (Ecuación 18, nota al pie 9 del paper):
$$\widehat{\text{Markup}}_m^\ell = \frac{p_m^\ell}{p_m^\ell - \min(\hat{\beta}_\eta^\ell, 1) \cdot \hat{\eta}_m^\ell}$$

La cota $\min(\hat{\beta}_\eta, 1)$ produce estimados conservadores.

**Precios de categoría por ciudad** `P_cat_46[m, j]`:
Construidos desde `P_46[producto]` (en pesos MXN, deflactados desde jun-2011)
ponderados por los shares de subproducto observados en la muestra final.
**No** se usa `exp(precios_matrix_ln)` que es un índice normalizado, no precios en pesos.

**Advertencia sobre resultados del Cuadro 8:**
Los $\beta_\eta$ estimados están sesgados hacia 1.0 en la mayoría de categorías
porque las elasticidades comprimidas generan $\eta_m \approx p_m$ para todas las ciudades,
reduciendo la variación identificadora de la regresión. Solo Pan (β=1.020 vs 1.477) y
Autobús foráneo (β=0.084 vs 0.081) replican razonablemente. Esta limitación es
consecuencia directa de la brecha de muestra y del solver de utilidad parcialmente convergente.


In [71]:
# =============================================================================
# 7.1. CENSO ECONÓMICO Y NORMALIZACIÓN DE VARIABLES
# =============================================================================

# 1. Lectura y unión del censo
df_general = pd.read_csv(DATA_DIR + 'censo_economico_2023.csv', skiprows=4).iloc[:-3]
df_fresnillo = pd.read_csv(DATA_DIR + 'censo_economico_fresnillo_2023.csv', skiprows=4).iloc[:-3]
df_censo = pd.concat([df_general, df_fresnillo], ignore_index=True)

# 2. Construcción de métricas relativas (Denominador: Unidades Económicas)
col_ue = 'UE Unidades económicas'
df_censo['UE'] = df_censo[col_ue]

métricas = {
    'empl_UE': 'H001A Personal ocupado total',
    'empl_remun_UE': 'H010A Personal remunerado total',
    'remun_UE': 'J000A Total de remuneraciones (millones de pesos)',
    'prod_UE': 'A111A Producción bruta total (millones de pesos)',
    'cons_interm_UE': 'A121A Consumo intermedio (millones de pesos)',
    'activos_UE': 'Q000A Acervo total de activos fijos (millones de pesos)',
    'deprec_UE': 'Q000B Depreciación total de activos fijos (millones de pesos)'
}
for nueva, original in métricas.items():
    df_censo[nueva] = pd.to_numeric(df_censo[original], errors='coerce') / df_censo[col_ue]

# 3. Limpieza de índice (Municipios)
def _normalizar_texto(texto):
    if pd.isna(texto): return ""
    texto = str(texto).strip().lower()
    for o, d in zip('áéíóúüñ.,;:', 'aeiouun    '):
        texto = texto.replace(o, d.strip())
    return texto

df_censo['Municipio'] = df_censo['Municipio'].fillna(df_censo['Entidad']).str.replace(r'^\d+\s*', '', regex=True)
#FIXME: Como tenemos solo 45 ciudades en conc (por solucionar, hacemos una trampa aqui quitando Ciudad acuna que no esta en conc)
df_censo = df_censo[df_censo['Municipio'] != 'Acuña']
df_censo.index = df_censo['Municipio'].map(_normalizar_texto)


In [72]:
# =============================================================================
# 7.2  Precios por ciudad en pesos MXN — desde df_precios_promedios y estructuras reales
# =============================================================================

# Obtener la lista única de ciudades desde el dataframe de precios
#FIXME: Como tenemos solo 45 ciudades en conc (por solucionar, hacemos una trampa aqui quitando Ciudad acuna que no esta en conc)
df_precios_promedios = df_precios_promedios[df_precios_promedios['City_name'] != 'Cd. Acuña, Coah.']
ciudades_disponibles = df_precios_promedios['City_name'].dropna().unique()
n_ciudades = len(ciudades_disponibles)

print(f"Ciudades detectadas en df_precios_promedios: {n_ciudades}")

# Función para calcular los pesos (shares) de los subproductos a nivel muestral
def calcular_shares_subproductos(gasto_prod_df, sub_prods):
    """Calcula el gasto promedio de cada subproducto respecto al total del grupo."""
    gastos_totales = np.array([gasto_prod_df[prod].sum() if prod in gasto_prod_df.columns else 0.0 for prod in sub_prods], dtype=float)
    total = gastos_totales.sum()
    return gastos_totales / total if total > 0 else np.ones(len(sub_prods)) / len(sub_prods)


# Inicializar matriz P_cat_46 dinámicamente basada en las ciudades reales
cats_n14 = list(CATEGORIAS.keys())
P_cat_46 = np.zeros((n_ciudades, len(cats_n14)))

# Pre-calcular shares globales usando gastos_por_producto
shares_dict = {}
for cat, sub_prods in CATEGORIAS.items():
    shares_dict[cat] = calcular_shares_subproductos(gastos_por_producto, sub_prods)

# Construir la matriz de precios ponderados por ciudad
for idx_c, ciudad in enumerate(ciudades_disponibles):
    df_ciudad = df_precios_promedios[df_precios_promedios['City_name'] == ciudad]
    
    # Mapeo de precios promedio por clase/producto en esta ciudad
    precios_ciudad = df_ciudad.set_index('Class')['Price_2022'].to_dict()
    
    for idx_cat, (cat, sub_prods) in enumerate(CATEGORIAS.items()):
        ws = shares_dict[cat]
        precio_ponderado = 0.0
        peso_acumulado = 0.0
        
        for prod, w in zip(sub_prods, ws):
            # Buscar precio del producto; si no existe en la ciudad, usar la media general del dataframe
            p_val = precios_ciudad.get(prod, df_precios_promedios[df_precios_promedios['Class'] == prod]['Price_2022'].mean())
            if pd.isna(p_val):
                p_val = 0.0
            precio_ponderado += p_val * w
        P_cat_46[idx_c, idx_cat] = precio_ponderado

print("\nPrecios por ciudad construidos exitosamente desde df_precios_promedios:")
for j, cat in enumerate(cats_n14):
    p = P_cat_46[:, j]
    print(f"  {cat:<15} min={p.min():.2f}  max={p.max():.2f}  media={p.mean():.2f}")

Ciudades detectadas en df_precios_promedios: 45

Precios por ciudad construidos exitosamente desde df_precios_promedios:
  Tortillas       min=12.06  max=29.95  media=21.61
  Pan             min=4.74  max=20.02  media=7.69
  Pollo y huevo   min=39.73  max=139.73  media=63.42
  Carne de res    min=73.46  max=216.38  media=131.44
  Carnes procesadas min=69.56  max=247.32  media=140.39
  Lácteos         min=36.19  max=88.49  media=65.60
  Frutas          min=29.56  max=57.21  media=39.49
  Verduras        min=22.94  max=41.83  media=30.56
  Bebidas         min=9.61  max=98.42  media=22.54
  Medicamentos    min=84.22  max=616.55  media=320.47
  Transporte foráneo min=35.99  max=2798.43  media=688.54
  Transporte aéreo min=828.45  max=60211.42  media=5510.73
  Materiales de construcción min=120.27  max=151.58  media=135.72


In [74]:
# ---------------------------------------------------------------
# 7.3  Estimación de markups — Adaptado dinámicamente a tus datos
# ---------------------------------------------------------------

#TODO: Checar si es normal lo de los 400% de Beta, parece que llega a saturacion el calculo

from datos_comparacion import (BETA_N_PAPER, T_V_PAPER, BETA_N_2014, T_V_2014, SOBREPRECIOS_PAPER, SOBREPRECIOS_2014)


# 1. Construcción y alineación de vars_costos desde df_censo
cols_costos = ['prod_UE', 'UE', 'empl_UE', 'remun_UE', 'cons_interm_UE', 'activos_UE', 'deprec_UE']

# Asegurar que la columna UE sea numérica
df_censo['UE'] = pd.to_numeric(df_censo['UE'], errors='coerce')

# Extraer matriz NumPy de costos (N_ciudades, 7)
vars_costos = df_censo[cols_costos].to_numpy()

cats_n = list(CATEGORIAS.keys())

# Convertir elasticidades a numpy en caso de que df_ciudades sea DataFrame
elastic_46 = df_ciudades.to_numpy() 

# Capturar las dimensiones reales de tus datos (ej. 45 o 46 ciudades)
n_ciudades, n_cats = P_cat_46.shape

beta_eta   = np.zeros(n_cats)
t_stat_eta = np.zeros(n_cats)
SE_eta     = np.zeros(n_cats)
markup_46  = np.zeros((n_ciudades, n_cats)) # Matriz adaptada a tus ciudades
factor_out = 1.5

for pfp in range(n_cats):
    precio_m  = P_cat_46[:, pfp]
    elastic_m = elastic_46[:, pfp]
    mask_neg  = elastic_m < 0
    if mask_neg.sum() < 5:
        markup_46[:, pfp] = 1.0;  continue

    eta_m = -precio_m[mask_neg] * (1.0 / elastic_m[mask_neg])
    X = np.column_stack([eta_m, vars_costos[mask_neg]])
    Y = precio_m[mask_neg]

    is_out = np.zeros(X.shape[0], dtype=bool)
    for col in range(X.shape[1]):
        q25,q75 = np.percentile(X[:,col],25), np.percentile(X[:,col],75)
        iqr = q75-q25
        is_out |= (X[:,col]<q25-factor_out*iqr)|(X[:,col]>q75+factor_out*iqr)
    X, Y = X[~is_out], Y[~is_out]
    N_   = X.shape[0]
    if N_ < 4:
        markup_46[:, pfp] = 1.0;  continue

    X  = np.column_stack([X, np.ones(N_)])
    try:
        betas = np.linalg.solve(X.T@X, X.T@Y)
    except:
        betas = np.linalg.lstsq(X, Y, rcond=None)[0]

    resid = Y - X@betas
    Sigma = (X.T@X)/N_
    Omega = (X.T@(X*resid[:,np.newaxis]**2))/N_
    try:
        V     = np.linalg.solve(Sigma, np.linalg.solve(Sigma, Omega).T).T
        se_b0 = np.sqrt(abs(V[0,0])/N_)
    except:
        se_b0 = np.nan

    b_eta = betas[0]
    t_eta = b_eta/se_b0 if (se_b0 and se_b0>0) else 0. #ici on multipliait par np.sqrt(N_)*
    beta_eta[pfp]   = b_eta
    t_stat_eta[pfp] = t_eta
    SE_eta[pfp]     = se_b0

    b_cap  = min(b_eta, 1.0)
    avg_e  = elastic_m[mask_neg].mean()
    
    # Bucle actualizado con n_ciudades en lugar de 46 fijo
    for m in range(n_ciudades):
        ciudad_actual = ciudades_disponibles[m]
        em       = elastic_m[m]
        eta_city = -precio_m[m]*(1./em if em<0 else 1./avg_e)
        cm       = precio_m[m] - b_cap*eta_city
        mk       = precio_m[m]/cm if cm>0 else 1.
        markup_46[m, pfp] = max(1., min(5., mk)) #comentado para ver los verdaderos valores
        # markup_46[m, pfp] = mk  # Guardar el markup real sin truncar

    sig = "***" if abs(t_eta)>=2.326 else ("**" if abs(t_eta)>=1.645 else "  ")
    print(f"  [{pfp+1:2d}] {cats_n[pfp]:<22} β={b_eta:>7.3f}  t={t_eta:>7.3f} {sig}  N={N_}")

print("\n=== CUADRO 8: PARÁMETROS DE PODER DE MERCADO β_η ===")

idx8 = [0,1,2,3,4,5,6,7,8,9,11,12,10]
print(f"{'Categoría':<22} {'β nuestro':>10} {'β paper':>8} {'β 2014':>8} {'t nuestro':>10} {'t paper':>8} {'t 2014':>8}")
print("-"*88)
for cat,pfp in zip(BETA_N_PAPER.keys(),idx8):
    b=beta_eta[pfp]; pb=BETA_N_PAPER[cat]; b2014=BETA_N_2014.get(cat, np.nan)
    t=t_stat_eta[pfp]; pt=T_V_PAPER[cat]; t2014=T_V_2014.get(cat, np.nan)
    sig="***" if abs(t)>=2.326 else ("**" if abs(t)>=1.645 else "  ")
    print(f"  {cat:<20} {b:>10.3f} {pb:>8.3f} {b2014:>8.3f} {t:>10.3f} {pt:>8.3f} {t2014:>8.3f} {sig}")

# Cuadro 9
print("\n=== CUADRO 9: SOBREPRECIOS ===")
print(f"{'Categoría':<22} {'Nuestro (%)':>12} {'Paper (%)':>10} {'2014 (%)':>10}")
print("-"*58)
sp_list = []
for cat, pfp in zip(BETA_N_PAPER.keys(), idx8):
    mv = markup_46[:,pfp][markup_46[:,pfp]>1]
    sp = (mv.mean()-1)*100 if len(mv)>0 else 0.
    sp_list.append(sp)
    sp_2014_val = SOBREPRECIOS_2014.get(cat, 0)
    sp_paper_val = SOBREPRECIOS_PAPER.get(cat, 0)
    print(f"  {cat:<20} {sp:>12.2f} {sp_paper_val:>10.2f} {sp_2014_val:>10.2f}")

sp_2014_mean = np.mean([SOBREPRECIOS_2014.get(cat, 0) for cat in BETA_N_PAPER.keys()])
print(f"\n  Promedio: {np.mean(sp_list):.2f}%  (Paper: 98.23%)  (2014: {sp_2014_mean:.2f}%)")


  [ 1] Tortillas              β=  0.547  t=  4.125 ***  N=33
  [ 2] Pan                    β=  0.966  t= 25.393 ***  N=31
  [ 3] Pollo y huevo          β=  0.996  t=357.940 ***  N=33
  [ 4] Carne de res           β=  0.953  t= 36.568 ***  N=35
  [ 5] Carnes procesadas      β=  0.973  t=168.025 ***  N=35
  [ 6] Lácteos                β=  0.998  t=106.552 ***  N=33
  [ 7] Frutas                 β=  0.982  t= 25.013 ***  N=34
  [ 8] Verduras               β=  0.964  t= 19.344 ***  N=34
  [ 9] Bebidas                β=  1.006  t=220.031 ***  N=34
  [10] Medicamentos           β=  0.950  t= 51.829 ***  N=32
  [11] Transporte foráneo     β=  1.220  t= 17.550 ***  N=32
  [12] Transporte aéreo       β=  1.026  t= 48.195 ***  N=31
  [13] Materiales de construcción β=  0.007  t=  0.513     N=35

=== CUADRO 8: PARÁMETROS DE PODER DE MERCADO β_η ===
Categoría               β nuestro  β paper   β 2014  t nuestro  t paper   t 2014
---------------------------------------------------------------------

## 8. Variación equivalente y pérdida de bienestar

### Sección 8 — Variación equivalente y pérdida de bienestar

**Variación equivalente** (Sección 2.1.4 del paper):
$$VE_h = C(\mathbf{p}_h^0, y_h(\mathbf{p}_h^1), z_h, \varepsilon_h) -
C(\mathbf{p}_h^0, y_h(\mathbf{p}_h^0), z_h, \varepsilon_h)$$

donde $\mathbf{p}_h^1$ = precios observados (con poder de mercado),
$\mathbf{p}_h^0$ = precios contrafactuales (sin markup, solo sectores sig. al 95%).

Implementación: $VE_h = \frac{C(\mathbf{p}^1, y^1, z, \varepsilon) - C(\mathbf{p}^0, y^1, z, \varepsilon)}{C(\mathbf{p}^1, y^1, z, \varepsilon)} \times x_h$

**Sectores significativos al 95%:** Los que tienen $\hat{\beta}_\eta > 0$ y
$t \geq 1.645$ (prueba de una cola). En nuestra estimación todos los sectores
resultan significativos (consecuencia de $\eta_m \approx p_m$).

**Resultados cuantitativos:**

| Resultado | Réplica | Paper | Diferencia |
|-----------|---------|-------|------------|
| VE media (pesos) | $3,970 | $1,497 | +2.65x |
| VE/ingreso media | 14.3% | 15.7% | -9% |
| Regresividad D1/D10 | 5.9x | 4.42x | +33% |
| Gini observado | 0.430 | 0.481 | -11% |
| Gini contrafactual | 0.406 | 0.446 | -9% |
| Reducción Gini | 5.6% | 7.3% | -23% |

**Interpretación:** Las proporciones (VE/ingreso %) replican bien el patrón
cualitativo porque VE e ingreso escalan juntos. El nivel en pesos está inflado
porque todos los sectores resultan significativos (vs 10 de 12 en el paper),
lo que amplía el vector de precios contrafactuales.

**Gini:** La reducción de 5.6% (vs 7.3% del paper) refleja la brecha de muestra.
Con 8,940 hogares el Gini observado es 0.430 (vs 0.481 del paper) — diferente
por la selección de muestra, no por error de metodología.


In [75]:
#FIXME: Números muy sketchy!
# ---------------------------------------------------------------
# 8.1  Variación equivalente — sin cambios respecto a v2
# ---------------------------------------------------------------
ciudades_disponibles = df_precios_promedios['City_name'].dropna().unique()
sig_95 = np.array([float(t>=1.645 and b>0)
                   for t,b in zip(t_stat_eta[:13], beta_eta[:13])])
cats_13 = list(CATEGORIAS.keys())
print("Sectores significativos al 95%:")
for cat,s in zip(cats_13,sig_95): print(f"  {cat:<22} {'✓' if s else '✗'}")

# 1. Crear el mapeo nombre_ciudad -> índice de fila m
ciudad_a_idx = {nombre: i for i, nombre in enumerate(ciudades_disponibles)}

# 2. Convertir los 57,507 hogares a sus índices numéricos de ciudad (0 a 44)
indices_hogares = conc['nombre_ciudad_cercana'].map(ciudad_a_idx).fillna(0).astype(int).to_numpy()

# 4. Asignar los markups a los 57,507 hogares
mk_hogar = markup_46[indices_hogares, :13]


p1_mat = np.asarray(precios_matrix_ln)
p0_mat = np.asarray(p1_mat - np.log(mk_hogar)*sig_95[np.newaxis,:])
Z_vars = np.asarray(Z_vars)
epsilon_matrix = np.asarray(epsilon_matrix)
w_matrix = np.asarray(w_matrix)
sg = np.asarray(sg)


def T_func(p,z): return 0.5*sum(float(z[l]*p@AZ_mats[l]@p) for l in range(9))
def S_func(p): return 0.5*float(p@B_mat@p)
def m_func(u,z): return b_poly@np.array([1.,u,u**2,u**3])+C_mat@z+D_mat@z*u
def easi_u(p,z,wh,ln_x):
    T=T_func(p,z); S=S_func(p)
    return (ln_x-float(p@wh)+T)/max(1.-S,1e-10)
def C_exp(p,u,z,eps):
    T=T_func(p,z); S=S_func(p); m=m_func(u,z)
    try: return math.exp(u+float(p@m)+T+S*u+float(p@eps))
    except OverflowError: return float('inf')

print("\nCalculando VE...")
VE = np.zeros(N)
ingreso = conc.loc[indices_activos, 'ing_cor'].to_numpy()
for i in range(N):
    p1=p1_mat[i]; p0=p0_mat[i]; z=Z_vars[i]
    eps=epsilon_matrix[i]; wh=w_matrix[i]; ln_x=math.log(sg[i])
    y1=easi_u(p1,z,wh,ln_x)
    Cp1=C_exp(p1,y1,z,eps); Cp0=C_exp(p0,y1,z,eps) #FIXME: demasiado cercanos
    if Cp1>0 and not math.isinf(Cp1) and not math.isinf(Cp0):
        VE[i]=((Cp1-Cp0)/Cp1)*sg[i]
    if i%2000==0: print(f"  {i}/{N}...")
VE=np.maximum(VE,0.)

print(f"\nVE media:   ${VE.mean():.0f}  (paper: $1,497)")
print(f"VE mediana: ${np.median(VE[VE>0]):.0f}")
VE_pct=(VE/np.where(ingreso>0,ingreso,np.nan))
print(f"VE/ingreso: {np.nanmean(VE_pct)*100:.1f}%  (paper: 15.7%)")


Sectores significativos al 95%:
  Tortillas              ✓
  Pan                    ✓
  Pollo y huevo          ✓
  Carne de res           ✓
  Carnes procesadas      ✓
  Lácteos                ✓
  Frutas                 ✓
  Verduras               ✓
  Bebidas                ✓
  Medicamentos           ✓
  Transporte foráneo     ✓
  Transporte aéreo       ✓
  Materiales de construcción ✗

Calculando VE...
  0/56355...
  2000/56355...
  4000/56355...
  6000/56355...
  8000/56355...
  10000/56355...
  12000/56355...
  14000/56355...
  16000/56355...
  18000/56355...
  20000/56355...
  22000/56355...
  24000/56355...
  26000/56355...
  28000/56355...
  30000/56355...
  32000/56355...
  34000/56355...
  36000/56355...
  38000/56355...
  40000/56355...
  42000/56355...
  44000/56355...
  46000/56355...
  48000/56355...
  50000/56355...
  52000/56355...
  54000/56355...
  56000/56355...

VE media:   $1532  (paper: $1,497)
VE mediana: $5567
VE/ingreso: 3.3%  (paper: 15.7%)


In [76]:
# ---------------------------------------------------------------
# 8.2  Cuadro 10 + Gini (Comparativo: Datos, Paper, Replica 2014)
# ---------------------------------------------------------------
from datos_comparacion import PAPER_M, PAPER_P, REPLICA_M, REPLICA_P

ing = ingreso
cuts = np.percentile(ing[ing > 0], np.arange(10, 101, 10))


def get_d(v):
  for d, c in enumerate(cuts, 1):
    if v <= c:
      return d
  return 10


decil_h = np.array([get_d(v) for v in ing])

# Aliases pour la clarté si nécessaire
paper_m, paper_p = PAPER_M, PAPER_P
replica_m, replica_p = REPLICA_M, REPLICA_P

print("=== CUADRO 10: PÉRDIDA DE BIENESTAR POR DECIL ===")
print(
    f"{'Decil':<6} {'VE($)':>8} {'Paper($)':>9} {'2014($)':>9} {'VE/Ing%':>9}"
    f" {'Paper%':>8} {'2014%':>8}"
)
print("-" * 63)

for d in range(1, 11):
  md = decil_h == d
  if md.sum() == 0:
    continue
  ve_d = VE[md]
  ing_d = ing[md]
  vm = ve_d.mean()
  vp = (ve_d / np.where(ing_d > 0, ing_d, np.nan)).mean() * 100
  print(
      f"  {d:<4} {vm:>8.0f} {paper_m[d-1]:>9.0f} {replica_m[d-1]:>9.0f}"
      f" {vp:>9.1f} {paper_p[d-1]:>8.1f} {replica_p[d-1]:>8.1f}"
  )

vt = VE.mean()
pt = np.nanmean(VE / np.where(ing > 0, ing, np.nan)) * 100
print(
    f"  {'Tot':<4} {vt:>8.0f} {paper_m[-1]:>9.0f} {replica_m[-1]:>9.0f}"
    f" {pt:>9.1f} {paper_p[-1]:>8.1f} {replica_p[-1]:>8.1f}"
)

# --- Regresividad ---
ve_d1 = VE[decil_h == 1]
ing_d1 = ing[decil_h == 1]
ve_d10 = VE[decil_h == 10]
ing_d10 = ing[decil_h == 10]
r1 = (ve_d1 / np.where(ing_d1 > 0, ing_d1, np.nan)).mean()
r10 = (ve_d10 / np.where(ing_d10 > 0, ing_d10, np.nan)).mean()

# Ratio D1/D10 para 2014
r_2014 = replica_p[0] / replica_p[9] if replica_p[9] != 0 else np.nan

print(
    f"\nRegresividad (D1/D10): {r1/r10:.2f}x  (paper: 4.42x | replica 2014:"
    f" {r_2014:.2f}x)"
)

# --- Índice de Gini ---
M = ing[ing > 0]
Ve = VE[ing > 0]
N_ = len(M)
Ms = np.sort(M)
rk = np.arange(1, N_ + 1)

G = (N_ + 1) / N_ - 2 * (((N_ + 1 - rk) * Ms).sum()) / (N_ * Ms.sum())
Mcf = np.sort(M + Ve)
Gcf = (N_ + 1) / N_ - 2 * (((N_ + 1 - rk) * Mcf).sum()) / (N_ * Mcf.sum())

print(f"\nGini observado:     {G:.3f}  (paper: 0.481)")
print(f"Gini contrafactual: {Gcf:.3f}  (paper: 0.446)")
print(f"Reducción:          {(G-Gcf)/G*100:.1f}%  (paper: 7.3%)")

=== CUADRO 10: PÉRDIDA DE BIENESTAR POR DECIL ===
Decil     VE($)  Paper($)   2014($)   VE/Ing%   Paper%    2014%
---------------------------------------------------------------
  1         808       841      2356       6.6     30.9     28.3
  2         986      1097      2883       4.3     23.6     20.7
  3        1189      1286      3378       4.0     21.4     18.5
  4        1312      1410      3570       3.5     18.9     16.0
  5        1591      1487      3752       3.6     16.7     14.0
  6        1594      1613      4150       3.0     15.1     13.0
  7        1759      1738      4291       2.7     13.6     11.2
  8        1891      1907      4634       2.4     11.9      9.6
  9        1945      2052      4791       1.9      9.5      7.4
  10       2242      2237      5934       1.3      5.7      4.8
  Tot      1532      1497      3974       3.3     15.7     14.3

Regresividad (D1/D10): 5.05x  (paper: 4.42x | replica 2014: 5.90x)

Gini observado:     0.411  (paper: 0.481)
Gini co

---

## Limitaciones de la réplica y camino hacia la actualización 2024

### Limitaciones identificadas

1. **Brecha de muestra (principal):** 8,940 hogares vs 15,586 del paper.
   Causa parcialmente no identificada — el Gauss posiblemente tiene filtros
   adicionales de muestra no completamente documentados en el paper.

2. **Convergencia del solver de utilidad:** Newton+damping converge en ~66% de
   hogares; el resto usa fallback. El Gauss usa `optmum()` con convergencia ~100%.
   Impacto: elasticidades comprimidas hacia 1.0 (MAE=0.207).

3. **Elasticidades Cuadro 4:** 5/13 dentro de ±0.15. Las regiones (Cuadro 5)
   replican exactamente (8/8 dentro de ±0.15) porque el error es sistemático
   (no diferenciado geográficamente).

4. **Markups y VE en pesos:** Sobreestimados (~3x) por las elasticidades comprimidas.
   El patrón cualitativo (regresividad, Gini) es correcto.

### Lo que está completamente implementado y listo para 2024

- ✅ Pipeline de precios: INPC × 46 ciudades × 61 subgéneros → precios en pesos
- ✅ Filtros de muestra ENIGH
- ✅ 12 categorías de gasto + índices Divisia
- ✅ 9 variables Z del hogar
- ✅ Sistema EASI: OLS iterado × 16, simetría, aditividad
- ✅ Utilidad indirecta exacta (Newton+damping)
- ✅ Elasticidades por ciudad × categoría
- ✅ OLS de markups con variables Censos Económicos
- ✅ Variación equivalente y descomposición por decil/región/Gini
